# NDI v1.4: Tasks 1-2 re-run with one row per date and p12 resolved

This notebook contains only the cells the v1.4 run depends on, so that **Runtime > Disconnect and delete runtime, then Run all** executes top to bottom in one session (G-0 asks once for the read-only token, hidden, not saved).

- Cells marked *v1, verbatim* are copied unchanged from `NDI_v1_3_forward.ipynb`, except three marked v1.4 edits: CELL 5 (p12 max heart rate read with `pd.to_numeric(..., errors="coerce")`; wellness no longer merged), CELL 6 (the wellness re-load and merge by UTC date removed: it duplicated 18 days), and the scratch-folder line (`results_regen_scratch_v1_4`).
- New data cells: **D-2** (every `days_df` is one row per consecutive calendar date, or the run stops) and **W-1** (wellness table under W1-W4, kept out of `days_df`).
- V12-1 ... V12-7 and X-2 are unchanged from v1.3 (X-2: folder and labels only). Changed: G-0 (private repository, checks out the tag), V12-0 (v1.4 config, tag, folders; F_SCALE asserted 1.0), V12-8 (last print), X-1 (v1.4 data facts), X-5 (v1.4 column), X-4 (v1.4 hand-in check).
- New: **X-6** (primary numbers without p12, only if p12 enters Table 1).

`results_v1.2/`, `results_v1.3/` and the frozen v1 archive are read, never written. Everything else is unchanged from v1.3: forward target with `min_periods = k`, purge, fold generator, SESOI, bootstrap, leak definitions and removal order.

## v1 cells (verbatim except the marked v1.4 edits in CELL 5, CELL 6 and the scratch-folder line), then D-2 and W-1

In [ ]:
# ================================================================
# CELL 1.0  CONFIGURATION  (edit paths to match your Drive)
# Brief Section 6: all parameters in one place, fixed seed
# ================================================================

#  Google Drive paths
PIPELINE_MODULE   = "/content/drive/MyDrive/ndi_project/ndi_pipeline.py"
DEPRESJON_DIR     = "/content/drive/MyDrive/ndi_project/depresjon-dataset/data"
PMDATA_DIR        = "/content/drive/MyDrive/ndi_project/pmdata"
PMDATA_OVERVIEW   = "/content/drive/MyDrive/ndi_project/pmdata/participant-overview.xlsx"
RESULTS_DIR       = "/content/drive/MyDrive/ndi_project/results"

#  Reproducibility seed (Brief Section 6)
RANDOM_SEED = 42

#  Mount Drive
from google.colab import drive
drive.mount("/content/drive")

import os, sys
os.makedirs(RESULTS_DIR, exist_ok=True)

# Add pipeline module to path
sys.path.insert(0, os.path.dirname(PIPELINE_MODULE))

print("Drive mounted.")
print("DEPRESJON dir exists:", os.path.isdir(DEPRESJON_DIR))
print("PMData    dir exists:", os.path.isdir(PMDATA_DIR))
print("Pipeline  exists    :", os.path.isfile(PIPELINE_MODULE))


In [ ]:
# Cell 1.1 --- results-capture helper (moved here so item 1 / item 3 / item 4 / item 7
#     calls earlier in the notebook can all use it) ---
import json, datetime

RESULTS_FINAL_DIR = os.path.join(os.path.dirname(RESULTS_DIR), "results_final")
os.makedirs(RESULTS_FINAL_DIR, exist_ok=True)
MANIFEST_PATH = os.path.join(RESULTS_FINAL_DIR, "manifest.json")
_manifest = []

def save_result(item_no, description, obj, filename, cell_label):
    path = os.path.join(RESULTS_FINAL_DIR, filename)
    if isinstance(obj, pd.DataFrame):
        obj.to_csv(path, index=False)
    else:
        with open(path, "w") as f:
            json.dump(obj, f, indent=2, default=str)
    _manifest.append(dict(item=item_no, description=description, cell=cell_label,
                           file=filename, saved_at=datetime.datetime.now().isoformat()))
    with open(MANIFEST_PATH, "w") as f:
        json.dump(_manifest, f, indent=2)
    print(f"[item {item_no}] saved -> {filename}")

In [ ]:
import shutil, json, os, numpy as np

_parent = os.path.dirname(RESULTS_DIR)
FROZEN_V1_DIR = os.path.join(_parent, "results_final_v1_frozen_backup")

if not os.path.exists(FROZEN_V1_DIR):
    _existing_manifest = json.load(open(os.path.join(RESULTS_FINAL_DIR, "manifest.json")))
    print(f"Current manifest.json has {len(_existing_manifest)} entries, items present: "
          f"{sorted(set(m['item'] for m in _existing_manifest))}")
    print("Confirm this looks like your complete v1 record before proceeding.")
    shutil.copytree(RESULTS_FINAL_DIR, FROZEN_V1_DIR)
    print(f"Backed up to {FROZEN_V1_DIR}")
else:
    print(f"Frozen backup already exists at {FROZEN_V1_DIR} -- not overwriting.")

RESULTS_FINAL_DIR = os.path.join(_parent, "results_regen_scratch_v1_4")   # v1.4: own scratch folder; this session writes here only
os.makedirs(RESULTS_FINAL_DIR, exist_ok=True)
MANIFEST_PATH = os.path.join(RESULTS_FINAL_DIR, "manifest.json")
_manifest = []

def benjamini_hochberg(pvals):   # identical to the CELL 14.9 definition -- redefined here
    pvals = np.asarray(pvals, dtype=float); n = len(pvals)
    order = np.argsort(pvals); q_raw = pvals[order] * n / (np.arange(n) + 1)
    q_sorted = np.minimum.accumulate(q_raw[::-1])[::-1]
    q = np.empty(n); q[order] = np.clip(q_sorted, 0, 1); return q

print("Frozen v1 backup:", FROZEN_V1_DIR, "| session writes ->", RESULTS_FINAL_DIR)

In [ ]:
# ================================================================
# CELL 2  INSTALL DEPENDENCIES + IMPORTS
# ================================================================
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["xgboost", "shap"]:
    install(pkg)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.tri as tri
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings("ignore")
np.random.seed(RANDOM_SEED)

import importlib.util, sys as _sys

spec = importlib.util.spec_from_file_location("ndi_pipeline", PIPELINE_MODULE)
ndi  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ndi)

print("ndi_pipeline loaded. Components available:")
for fn in ["neutrosophic_decompose","blocked_tscv","run_baselines",
           "lambda_grid_search","shap_attribution"]:
    print(f"  ndi.{fn}: {'OK' if hasattr(ndi, fn) else 'MISSING'}")


In [ ]:
# ================================================================
# CELL 3 -- LOAD DEPRESJON DATASET
# Structure: depresjon-dataset/data/condition/condition_1.csv ...
#            depresjon-dataset/data/control/control_1.csv ...
#            depresjon-dataset/data/scores.csv
# scores.csv: number=condition_1/condition_2/control_1 etc.
# ================================================================
import os, glob
import pandas as pd
import numpy as np

def load_depresjon(base_dir):
    scores_path = os.path.join(base_dir, "scores.csv")
    scores      = pd.read_csv(scores_path)
    scores.columns = [c.strip().lower() for c in scores.columns]
    print("scores.csv columns:", list(scores.columns))
    print(scores.head(3))

    # Build lookup: pid_string -> (madrs1, madrs2)
    madrs_lookup = {}
    for _, row in scores.iterrows():
        key = str(row["number"]).strip()
        m1  = float(row["madrs1"]) if str(row.get("madrs1","NA")).strip() not in ["NA","nan",""] else np.nan
        m2  = float(row["madrs2"]) if str(row.get("madrs2","NA")).strip() not in ["NA","nan",""] else np.nan
        madrs_lookup[key] = (m1, m2)

    participants = []

    for group in ["condition", "control"]:
        folder = os.path.join(base_dir, group)
        csvs   = sorted(glob.glob(os.path.join(folder, "*.csv")))
        print(f"\n{group}: {len(csvs)} files found")

        for csv_path in csvs:
            pid = os.path.splitext(os.path.basename(csv_path))[0]
            try:
                df = pd.read_csv(csv_path)
                df.columns = [c.strip().lower() for c in df.columns]

                if "timestamp" not in df.columns or "activity" not in df.columns:
                    print(f"  SKIP {pid}: columns={list(df.columns)}")
                    continue

                madrs1, madrs2 = madrs_lookup.get(pid, (np.nan, np.nan))
                madrs_mean     = np.nanmean([madrs1, madrs2])

                daily = ndi.extract_daily_features(df)

                if len(daily) < 5:
                    print(f"  SKIP {pid}: only {len(daily)} days")
                    continue

                participants.append(dict(
                    pid          = pid,
                    group        = group,
                    is_depressed = int(group == "condition"),
                    madrs1       = madrs1,
                    madrs2       = madrs2,
                    madrs_mean   = madrs_mean,
                    n_days       = len(daily),
                    days_df      = daily,
                ))
                print(f"  OK {pid}: {len(daily)} days  MADRS={madrs_mean:.0f}" if not np.isnan(madrs_mean)
                      else f"  OK {pid}: {len(daily)} days  MADRS=N/A")

            except Exception as e:
                print(f"  ERROR {pid}: {e}")

    return participants

depresjon_participants = load_depresjon(DEPRESJON_DIR)

condition_p = [p for p in depresjon_participants if p["group"] == "condition"]
control_p   = [p for p in depresjon_participants if p["group"] == "control"]
print("  NOTE: madrs_mean = NaN for control participants — expected, DEPRESJON administered MADRS to condition group only.")
print(f"\nLoaded total: {len(depresjon_participants)} participants")
print(f"  Condition (depressed): {len(condition_p)}")
print(f"  Control   (healthy)  : {len(control_p)}")
if condition_p:
    print(f"  Median days condition: {np.median([p['n_days'] for p in condition_p]):.0f}")
    valid_madrs = [p['madrs_mean'] for p in condition_p if not np.isnan(p['madrs_mean'])]
    if valid_madrs:
        print(f"  MADRS range: {min(valid_madrs):.0f} to {max(valid_madrs):.0f}")
if control_p:
    print(f"  Median days control  : {np.median([p['n_days'] for p in control_p]):.0f}")

In [ ]:
# ================================================================
# CELL 4  BUILD DEPRESJON FEATURE MATRIX
# FIX 2: neutro_df built here is FOR VISUALISATION ONLY.
# All CV metrics rebuild features per-fold in Cell 8.
# FIX 3: calls ndi.build_neutrosophic_features (module is
# authoritative). Inline decomposition deleted.
# ================================================================

FEAT_NAMES = ndi.DEPRESJON_FEATURE_NAMES
DOMAIN_MAP = ndi.DEPRESJON_DOMAIN_MAP

def build_participant_neutrosophic(p):
    """
    For one participant: extract feature matrix, apply neutrosophic
    decomposition using full-series train_mask (visualisation only).
    WARNING: Do NOT use neutro_df from this function in any CV result.
    All CV evaluation rebuilds features inside each fold (Cell 8).
    """
    df = p["days_df"].copy().reset_index(drop=True)
    for col in FEAT_NAMES:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = 0.0

    X = df[FEAT_NAMES].values
    n = len(X)

    # Full-series train_mask: all True — avoids the ValueError in
    # build_neutrosophic_features while making clear this is in-sample.
    # VISUALISATION ONLY — do not use for any headline metric.
    train_mask_full = np.ones(n, dtype=bool)
    neutro_df = ndi.build_neutrosophic_features(
        X, FEAT_NAMES, train_mask=train_mask_full)

    result = pd.concat([df[["date"]].reset_index(drop=True),
                        neutro_df.reset_index(drop=True)], axis=1)
    result["pid"]          = p["pid"]
    result["group"]        = p["group"]
    result["is_depressed"] = p["is_depressed"]
    result["madrs_mean"]   = p["madrs_mean"]
    for col in FEAT_NAMES:
        result[col] = df[col].values
    return result

print("Computing neutrosophic features (visualisation only)...")
neutro_frames = []
for p in depresjon_participants:
    try:
        nf = build_participant_neutrosophic(p)
        neutro_frames.append(nf)
        p["neutro_df"] = nf
    except Exception as e:
        print(f"  ERROR {p['pid']}: {e}")

depresjon_all = pd.concat(neutro_frames, ignore_index=True)
print(f"Total rows: {len(depresjon_all)}")
print("NOTE: neutro_df is for visualisation only. CV metrics use per-fold features.")

In [ ]:
# ================================================================
# CELL 5 -- LOAD PMDATA DATASET   [v1.4: two changes, marked 'v1.4' below]
# Actual structure:
#   pmdata/p01/fitbit/heart_rate.json
#   pmdata/p01/fitbit/resting_heart_rate.json
#   pmdata/p01/fitbit/sleep.json
#   pmdata/p01/fitbit/sleep_score.csv
#   pmdata/p01/fitbit/steps.json
#   pmdata/p01/fitbit/lightly_active_minutes.json
#   pmdata/p01/fitbit/moderately_active_minutes.json
#   pmdata/p01/fitbit/very_active_minutes.json
#   pmdata/p01/fitbit/sedentary_minutes.json
#   pmdata/p01/pmsys/wellness.csv    <- daily mood/wellness self-report
#   pmdata/p01/pmsys/srpe.csv
#   pmdata/participant-overview.xlsx <- max HR, age, gender per participant
# ================================================================
import os, glob, json as _json
import pandas as pd
import numpy as np
import openpyxl  # for reading xlsx

#  Load participant overview
try:
    overview = pd.read_excel(PMDATA_OVERVIEW, header=1)
    overview.columns = [str(c).strip().lower().replace(" ","_")
                        for c in overview.columns]
    overview = overview.dropna(subset=[overview.columns[0]])
    print("participant-overview columns:", list(overview.columns))
    print(overview.head(3))
except Exception as e:
    print(f"Could not load participant-overview: {e}")
    overview = pd.DataFrame()

#  Helper: load one JSON Fitbit file
def load_fitbit_json(path):
    """Load a Fitbit JSON file. Returns list of {dateTime, value} dicts."""
    try:
        with open(path) as f:
            data = _json.load(f)
        if isinstance(data, list):
            return data
        return []
    except Exception:
        return []

def fitbit_to_daily(entries, value_key="value", agg="mean"):
    """Aggregate minute/entry-level Fitbit data to daily values."""
    rows = {}
    for entry in entries:
        date = str(entry.get("dateTime",""))[:10]
        val  = entry.get(value_key, entry.get("value", None))
        if date and val is not None:
            try:
                val = float(val)
                rows.setdefault(date, []).append(val)
            except (ValueError, TypeError):
                pass
    if agg == "mean":
        return {d: np.mean(v) for d,v in rows.items()}
    elif agg == "sum":
        return {d: np.sum(v) for d,v in rows.items()}
    elif agg == "max":
        return {d: np.max(v) for d,v in rows.items()}
    return rows

#  Helper: load resting heart rate JSON
def load_resting_hr(path):
    """resting_heart_rate.json has nested value.value structure."""
    entries = load_fitbit_json(path)
    rows = {}
    for entry in entries:
        date = str(entry.get("dateTime",""))[:10]
        val  = entry.get("value",{})
        if isinstance(val, dict):
            rhr = val.get("value", val.get("restingHeartRate", None))
        else:
            rhr = val
        if date and rhr is not None:
            try:
                rows[date] = float(rhr)
            except (ValueError, TypeError):
                pass
    return rows

#  Helper: load sleep.json
def load_sleep(path):
    """Extract sleep efficiency and minutes asleep from sleep.json."""
    entries = load_fitbit_json(path)
    eff_rows = {}; min_rows = {}
    for entry in entries:
        # Each entry is one sleep session
        date = str(entry.get("dateOfSleep", entry.get("dateTime","")))[:10]
        eff  = entry.get("efficiency", None)
        mins = entry.get("minutesAsleep", None)
        if date and eff is not None:
            try:
                eff_rows[date] = float(eff)
            except (ValueError, TypeError):
                pass
        if date and mins is not None:
            try:
                min_rows[date] = float(mins)
            except (ValueError, TypeError):
                pass
    return eff_rows, min_rows

#  Helper: load sleep_score.csv
def load_sleep_score(path):
    """sleep_score.csv: timestamp, overall_score, composition_score, ..."""
    try:
        df = pd.read_csv(path)
        df.columns = [c.strip().lower() for c in df.columns]
        # Timestamp column
        ts_col = next((c for c in df.columns if "timestamp" in c or "date" in c), None)
        score_col = next((c for c in df.columns if "overall" in c or "score" in c), None)
        if ts_col and score_col:
            df["date"] = pd.to_datetime(df[ts_col]).dt.strftime("%Y-%m-%d")
            return dict(zip(df["date"], pd.to_numeric(df[score_col], errors="coerce")))
    except Exception:
        pass
    return {}

#  Helper: load wellness.csv
def load_wellness(path):
    """
    pmsys/wellness.csv: daily self-report.
    Columns typically: date, fatigue, mood, readiness, sleep_duration,
                       sleep_quality, stress, soreness
    """
    try:
        df = pd.read_csv(path)
        df.columns = [c.strip().lower().replace(" ","_") for c in df.columns]
        if "date" not in df.columns:
            return pd.DataFrame()
        df["date"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d")
        return df
    except Exception:
        return pd.DataFrame()

#  Helper: load googledocs/reporting.csv
def load_reporting(path):
    """
    reporting.csv: date, timestamp, meals, weight, glasses_of_fluid, alcohol
    """
    try:
        df = pd.read_csv(path)
        df.columns = [c.strip().lower().replace(" ","_") for c in df.columns]
        date_col = next((c for c in df.columns if "date" in c), None)
        if date_col:
            df["date"] = pd.to_datetime(df[date_col]).dt.strftime("%Y-%m-%d")
            return df
    except Exception:
        pass
    return pd.DataFrame()

#  Main loader
def load_pmdata_participant(pid_dir):
    """
    Load all data for one PMData participant.
    Returns a daily DataFrame with all available signals.
    """
    fitbit_dir    = os.path.join(pid_dir, "fitbit")
    pmsys_dir     = os.path.join(pid_dir, "pmsys")
    googledocs_dir= os.path.join(pid_dir, "googledocs")

    daily = {}  # date -> feature dict

    def add_series(series_dict, col_name):
        for date, val in series_dict.items():
            daily.setdefault(date, {})[col_name] = val

    # Resting heart rate (most reliable HR signal for daily level)
    rhr_path = os.path.join(fitbit_dir, "resting_heart_rate.json")
    if os.path.isfile(rhr_path):
        add_series(load_resting_hr(rhr_path), "resting_hr")

    # Steps
    steps_path = os.path.join(fitbit_dir, "steps.json")
    if os.path.isfile(steps_path):
        add_series(fitbit_to_daily(load_fitbit_json(steps_path), agg="sum"), "steps")

    # Activity minutes
    for fname, col in [
        ("lightly_active_minutes.json",    "lightly_active_min"),
        ("moderately_active_minutes.json", "moderately_active_min"),
        ("very_active_minutes.json",       "very_active_min"),
        ("sedentary_minutes.json",         "sedentary_min"),
    ]:
        p = os.path.join(fitbit_dir, fname)
        if os.path.isfile(p):
            add_series(fitbit_to_daily(load_fitbit_json(p), agg="sum"), col)

    # Sleep
    sleep_path = os.path.join(fitbit_dir, "sleep.json")
    if os.path.isfile(sleep_path):
        eff, mins = load_sleep(sleep_path)
        add_series(eff,  "sleep_efficiency")
        add_series(mins, "sleep_minutes")

    # Sleep score
    ss_path = os.path.join(fitbit_dir, "sleep_score.csv")
    if os.path.isfile(ss_path):
        add_series(load_sleep_score(ss_path), "sleep_score")

    if not daily:
        return pd.DataFrame()

    # Build base daily DataFrame
    df = pd.DataFrame.from_dict(daily, orient="index")
    df.index.name = "date"
    df = df.reset_index()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)

    # Active minutes total
    act_cols = [c for c in ["lightly_active_min","moderately_active_min","very_active_min"] if c in df.columns]
    if act_cols:
        df["active_minutes_total"] = df[act_cols].sum(axis=1)

    # v1.4: wellness is NOT merged into days_df (v1.4 memo, Section 4). It is built as a separate
    # table under rules W1-W4 in cell W-1 and joined only when the Task 3 target is built.

    # Merge reporting (weight, fluid)
    r_path = os.path.join(googledocs_dir, "reporting.csv")
    if os.path.isfile(r_path):
        rep = load_reporting(r_path)
        if not rep.empty:
            rep["date"] = pd.to_datetime(rep["date"])
            rep_agg = rep.groupby("date").agg(
                weight         = ("weight", "mean"),
                glasses_of_fluid=("glasses_of_fluid","mean") if "glasses_of_fluid" in rep.columns else ("date","count"),
            ).reset_index()
            df = df.merge(rep_agg, on="date", how="left")

    df["date"] = df["date"].dt.strftime("%Y-%m-%d")
    return df


#  Load all participants
pmdata_participants = []
participant_dirs = sorted([
    os.path.join(PMDATA_DIR, d)
    for d in os.listdir(PMDATA_DIR)
    if os.path.isdir(os.path.join(PMDATA_DIR, d)) and d.startswith("p")
])
print(f"Found {len(participant_dirs)} participant directories")

for pdir in participant_dirs:
    pid = os.path.basename(pdir)
    try:
        daily = load_pmdata_participant(pdir)
        if len(daily) < 7:
            print(f"  SKIP {pid}: only {len(daily)} days")
            continue

        # Get max HR from overview
        max_hr = np.nan
        if not overview.empty:
            hr_col = next((c for c in overview.columns if "heart" in c and "max" in c), None)
            id_col = next((c for c in overview.columns if "participant" in c or "id" in c), None)
            if hr_col and id_col:
                row = overview[overview[id_col].astype(str).str.lower().str.strip() == pid.lower()]
                if not row.empty:
                    max_hr = pd.to_numeric(row.iloc[0][hr_col], errors="coerce")   # v1.4: p12's cell is ' ' (v1.4 memo, Sec. 2); not a model feature

        pmdata_participants.append(dict(
            pid       = pid,
            group     = "pmdata",
            is_depressed = 0,
            max_hr    = max_hr,
            n_days    = len(daily),
            days_df   = daily,
        ))
        cols_found = [c for c in daily.columns if c != "date"]
        print(f"  OK {pid}: {len(daily)} days  max_HR={max_hr:.0f}  signals={cols_found[:6]}")
    except Exception as e:
        print(f"  ERROR {pid}: {e}")

print(f"\nPMData: {len(pmdata_participants)} participants loaded")
if pmdata_participants:
    print(f"  Median days: {np.median([p['n_days'] for p in pmdata_participants]):.0f}")
    all_cols = set()
    for p in pmdata_participants:
        all_cols.update(p['days_df'].columns.tolist())
    print(f"  All available columns: {sorted(all_cols)}")

In [ ]:
# ================================================================
# CELL 6  PMData FEATURE ENGINEERING + WELLNESS MERGE   [v1.4: wellness merge removed, marked 'v1.4']
# Defines feature set, builds neutrosophic features, and crucially
# merges wellness scores as the external target column.
# ================================================================
import numpy as np
import pandas as pd

if not pmdata_participants:
    print("No PMData participants loaded. Check PMDATA_DIR.")
    pmdata_all = pd.DataFrame()
    PMDATA_FEATURE_NAMES = []
    PMDATA_DOMAIN_MAP    = {}
    PMDATA_WELLNESS      = []
else:
    # Collect all columns across participants
    sample_cols = set()
    for p in pmdata_participants:
        sample_cols.update(p["days_df"].columns.tolist())
    sample_cols.discard("date")

    PMDATA_PHYS = [c for c in [
        "resting_hr","steps","active_minutes_total",
        "lightly_active_min","moderately_active_min",
        "very_active_min","sedentary_min"
    ] if c in sample_cols]

    PMDATA_SLEEP = [c for c in [
        "sleep_efficiency","sleep_minutes","sleep_score"
    ] if c in sample_cols]

    # Wellness columns — these become the EXTERNAL TARGET for lambda search
    PMDATA_WELLNESS = [c for c in [
        "fatigue","mood","readiness","sleep_quality","stress","soreness"
    ] if c in sample_cols]

    PMDATA_FEATURE_NAMES = PMDATA_PHYS + PMDATA_SLEEP
    PMDATA_DOMAIN_MAP    = {"physiology": PMDATA_PHYS, "sleep": PMDATA_SLEEP}

    print("PMData physiology features:", PMDATA_PHYS)
    print("PMData sleep features:     ", PMDATA_SLEEP)
    print("PMData wellness/target cols:", PMDATA_WELLNESS)
    print("Total NDI features:        ", len(PMDATA_FEATURE_NAMES))

    # v1.4: the wellness re-load and merge that stood here (UTC date, one row per REPORT) is removed: it is what
    # duplicated 18 days in v1-v1.3 (v1.4 memo). days_df stays one row per Fitbit date; wellness is built in W-1.
    print("v1.4: wellness kept out of days_df (separate table, cell W-1)")

    # Build neutrosophic features for each participant
    print("\nComputing neutrosophic features for PMData participants...")
    pmdata_frames = []
    for p in pmdata_participants:
        try:
            df = p["days_df"].copy().reset_index(drop=True)

            for col in PMDATA_FEATURE_NAMES:
                if col not in df.columns:
                    df[col] = np.nan
                df[col] = pd.to_numeric(df[col], errors="coerce")
                df[col] = df[col].fillna(df[col].median())

            X        = df[PMDATA_FEATURE_NAMES].values
            train_mask_full = np.ones(len(X), dtype=bool)
            neutro   = ndi.build_neutrosophic_features(X, PMDATA_FEATURE_NAMES,
                                            train_mask=train_mask_full)
            result   = pd.concat([df[["date"]].reset_index(drop=True),
                                   neutro.reset_index(drop=True)], axis=1)
            result["pid"]    = p["pid"]
            result["max_hr"] = p["max_hr"]

            # Keep raw features AND wellness columns
            for col in PMDATA_FEATURE_NAMES + PMDATA_WELLNESS:
                if col in df.columns:
                    result[col] = df[col].values

            pmdata_frames.append(result)
            p["neutro_df"] = result

            well_coverage = {c: df[c].notna().sum() for c in PMDATA_WELLNESS
                             if c in df.columns}
            print(f"  {p['pid']}: {len(result)} days  "
                  f"NDI={result['NDI'].mean():.4f}  "
                  f"wellness={well_coverage}")

        except Exception as e:
            print(f"  ERROR {p['pid']}: {e}")

    if pmdata_frames:
        pmdata_all = pd.concat(pmdata_frames, ignore_index=True)
        print(f"\nPMData total rows: {len(pmdata_all)}")
        print(f"NDI range: [{pmdata_all['NDI'].min():.4f},"
              f"{pmdata_all['NDI'].max():.4f}]")
        for wcol in PMDATA_WELLNESS:
            if wcol in pmdata_all.columns:
                vals = pmdata_all[wcol].dropna()
                if len(vals) > 0:
                    print(f"  {wcol}: mean={vals.mean():.2f}  "
                          f"range=[{vals.min():.0f},{vals.max():.0f}]  "
                          f"n={len(vals)}")
    else:
        print("No PMData frames built.")
        pmdata_all           = pd.DataFrame()
        PMDATA_FEATURE_NAMES = []
        PMDATA_DOMAIN_MAP    = {}
        PMDATA_WELLNESS      = []

In [ ]:
# D-2 (v1.4) One row per local calendar date: every days_df must have unique, consecutive dates (v1.4 memo, Section 4)
# days_df comes straight from the Fitbit daily files (CELL 5); wellness is not merged into it any more (CELL 5, CELL 6).
# Any duplicated, unparseable or missing date stops the run here, with the offending dates printed.
import numpy as np, pandas as pd
_WELL_COLS = {"fatigue", "mood", "readiness", "sleep_quality", "stress", "soreness", "effective_time_frame"}
_rows, _bad = [], []
for p in pmdata_participants:
    d = pd.to_datetime(p["days_df"]["date"], errors="coerce")
    dd = d.dropna()
    steps = dd.diff().dt.days.dropna()
    gaps = [(str(a.date()), str(b.date())) for a, b, s in zip(dd.iloc[:-1], dd.iloc[1:], steps) if s != 1]
    leaked = sorted(_WELL_COLS & set(p["days_df"].columns))
    _rows.append(dict(pid=p["pid"], rows=len(d), first_date=str(dd.min().date()), last_date=str(dd.max().date()),
                      unparseable_dates=int(d.isna().sum()), duplicated_dates=int(d.duplicated().sum()),
                      non_consecutive_steps=len(gaps), wellness_columns_in_days_df=len(leaked)))
    if d.isna().any() or d.duplicated().any() or gaps or leaked:
        _bad.append((p["pid"], int(d.isna().sum()), int(d.duplicated().sum()), gaps[:10], leaked))
DATE_CHECK = pd.DataFrame(_rows)
print(DATE_CHECK.to_string(index=False))
if _bad:
    for pid, n_nat, n_dup, g, lk in _bad:
        print(f"  {pid}: unparseable {n_nat}, duplicated {n_dup}, steps != 1 day (previous, next): {g}, wellness columns: {lk}")
    raise RuntimeError("STOP: days_df is not one row per consecutive calendar date (v1.4 memo, Section 4). "
                       "Send the lines above to ma'am; do not fix by hand.")
print(f"D-2 PASSED: {len(DATE_CHECK)} participants, {int(DATE_CHECK['rows'].sum())} rows, "
      "every date unique and consecutive, no wellness column in days_df")

In [ ]:
# W-1 (v1.4) Wellness table under W1-W4, kept OUT of days_df (v1.4 memo, Sections 3-4)
# One row per participant and local date; joined only when the Task 3 target is built. Tasks 1-2 do not use it.
# Same functions as W-0 in NDI_v1_4_prechecks.ipynb. Built here, early, so a problem stops the run in minutes;
# the files are written by X-1 into results_v1.4.
import os
import numpy as np, pandas as pd

# ---- Wellness rules W1-W4 (v1.4 memo, Section 3). Identical in NDI_v1_4_prechecks.ipynb (W-0) and NDI_v1_4.ipynb (W-1).
WELLNESS_TZ = "Europe/Oslo"
W_ITEMS_1_5 = ["mood", "stress", "fatigue", "sleep_quality", "soreness"]   # scored 1-5: 0 = missing (W2)
W_ITEMS_ALL = W_ITEMS_1_5 + ["readiness"]                                  # readiness runs 0-10: 0 is valid
W_RANGE = {**{c: (1, 5) for c in W_ITEMS_1_5}, "readiness": (0, 10)}

def load_wellness_reports(pmdata_dir, pid):
    """Every report in pmsys/wellness.csv, untouched, with its UTC and local (W1) calendar date."""
    path = os.path.join(pmdata_dir, pid, "pmsys", "wellness.csv")
    if not os.path.isfile(path):
        return None
    w = pd.read_csv(path)
    w.columns = [c.strip().lower().replace(" ", "_") for c in w.columns]
    raw = w["effective_time_frame"].astype(str).str.strip()
    ts = pd.to_datetime(raw, errors="coerce", utc=True)
    rep = pd.DataFrame({"pid": pid, "timestamp": raw.values, "ts_utc": ts.values})
    rep["ts_utc"] = pd.to_datetime(rep["ts_utc"], utc=True)
    rep["tz_aware"] = raw.str.contains(r"(?:Z|[+-]\d\d:?\d\d)$", regex=True).values
    rep["utc_date"] = rep["ts_utc"].dt.strftime("%Y-%m-%d")
    rep["local_date"] = rep["ts_utc"].dt.tz_convert(WELLNESS_TZ).dt.strftime("%Y-%m-%d")      # W1
    for c in W_ITEMS_ALL:
        rep[c] = pd.to_numeric(w[c], errors="coerce").values if c in w.columns else np.nan
    return rep

def apply_wellness_rules(rep):
    """W1-W4 for one participant. Returns (daily table: one row per local date, log dict)."""
    pid = rep["pid"].iloc[0] if len(rep) else None
    bad_ts = rep["ts_utc"].isna() | ~rep["tz_aware"]
    r = rep[~bad_ts].copy()
    log = dict(pid=pid, reports=len(rep), timestamps_unparseable_or_without_timezone=int(bad_ts.sum()),
               reports_moved_by_W1=int((r["utc_date"] != r["local_date"]).sum()))
    for c in W_ITEMS_ALL:                                   # anything other than 0 outside the scale
        lo, hi = W_RANGE[c]
        v = r[c]
        log[f"out_of_range_{c}"] = int((v.notna() & (v != 0) & ((v < lo) | (v > hi) | (v != v.round()))).sum())
    for c in W_ITEMS_1_5:
        log[f"zeros_{c}"] = int((r[c] == 0).sum())
    blank = (r[W_ITEMS_1_5] == 0).all(axis=1)              # W3: all five 1-5 items are 0 -> blank submission
    log["reports_dropped_W3"] = int(blank.sum())
    r = r[~blank].copy()
    for c in W_ITEMS_1_5:                                   # W2: 0 = missing on the 1-5 items
        log[f"zeros_set_missing_W2_{c}"] = int((r[c] == 0).sum())
        r.loc[r[c] == 0, c] = np.nan
    r = r.sort_values("ts_utc", kind="mergesort")
    g = r.groupby("local_date", sort=True)
    daily = g[W_ITEMS_ALL].last()                           # W4: last valid value of the day, item by item
    daily.insert(0, "reports_that_day", g.size())
    daily = daily.reset_index().rename(columns={"local_date": "date"})
    daily.insert(0, "pid", pid)
    log["local_days_with_a_report"] = len(daily)
    log["days_with_2plus_reports_W4"] = int((daily["reports_that_day"] >= 2).sum())
    for c in W_ITEMS_ALL:
        nvalid = g[c].count()
        nuniq = g[c].nunique()
        log[f"W4_days_choosing_{c}"] = int((nvalid >= 2).sum())         # days where W4 picked among valid values
        log[f"W4_days_values_differ_{c}"] = int((nuniq >= 2).sum())     # ... and the values disagreed
    return daily, log

def build_wellness_tables(pmdata_dir, pids):
    tables, logs = {}, []
    for pid in pids:
        rep = load_wellness_reports(pmdata_dir, pid)
        if rep is None or rep.empty:
            logs.append(dict(pid=pid, reports=0))
            continue
        tables[pid], lg = apply_wellness_rules(rep)
        logs.append(lg)
    log = pd.DataFrame(logs).fillna(0)
    for c in log.columns:
        if c != "pid":
            log[c] = log[c].astype(int)
    return tables, log

def wellness_zero_table(log):
    """Deliverable 2: zeros per participant and 1-5 item (all reports), blank reports W3 drops, zero share."""
    z = log[["pid", "reports"] + [f"zeros_{c}" for c in W_ITEMS_1_5] + ["reports_dropped_W3"]].copy()
    z["zeros_total"] = z[[f"zeros_{c}" for c in W_ITEMS_1_5]].sum(axis=1)
    z["zero_share_of_item_values"] = (z["zeros_total"] / (5 * z["reports"]).where(z["reports"] > 0)).round(3)
    z["zeros_left_after_W3"] = z["zeros_total"] - 5 * z["reports_dropped_W3"]
    return z

_pids = sorted(d for d in os.listdir(PMDATA_DIR) if os.path.isdir(os.path.join(PMDATA_DIR, d)) and d.startswith("p"))
wellness_tables, wellness_log = build_wellness_tables(PMDATA_DIR, _pids)
wellness_daily = pd.concat(list(wellness_tables.values()), ignore_index=True) if wellness_tables else pd.DataFrame()
wellness_zeros = wellness_zero_table(wellness_log)
_chk = ["timestamps_unparseable_or_without_timezone"] + [f"out_of_range_{c}" for c in W_ITEMS_ALL]
_bad = wellness_log[wellness_log[_chk].sum(axis=1) > 0]
if not _bad.empty:
    print(_bad[["pid"] + _chk].to_string(index=False))
    raise RuntimeError("STOP: wellness timestamps or values the W1-W4 rules do not cover (see above). Report to ma'am.")
assert not wellness_daily.duplicated(["pid", "date"]).any(), "wellness table is not one row per local date"
_fit_dates = {p["pid"]: set(pd.to_datetime(p["days_df"]["date"]).dt.strftime("%Y-%m-%d")) for p in pmdata_participants}
wellness_log["local_days_outside_fitbit_dates"] = [
    int((~wellness_daily.loc[wellness_daily["pid"] == pid, "date"].isin(_fit_dates[pid])).sum()) if pid in _fit_dates else np.nan
    for pid in wellness_log["pid"]]
print(f"W-1: {len(wellness_daily)} participant-days from {int(wellness_log['reports'].sum())} reports | "
      f"W1 moved {int(wellness_log['reports_moved_by_W1'].sum())} reports | W3 dropped {int(wellness_log['reports_dropped_W3'].sum())} | "
      f"W2 set {int(wellness_zeros['zeros_left_after_W3'].sum())} zeros to missing | "
      f"days with 2+ reports (W4): {int(wellness_log['days_with_2plus_reports_W4'].sum())}, "
      f"mood values differing on {int(wellness_log['W4_days_values_differ_mood'].sum())} of them")

In [ ]:
# ================================================================
# CELL 6.5  F_SCALE PRE-SPECIFICATION SWEEP  (follow-up note, "The fix")
# Run this BEFORE Cell 7 and BEFORE inspecting any CV/rho result.
#
# Pre-specified rule (fixed in advance; do not revise after seeing
# the numbers): sweep SCALE over {1, 2, 3, 4, 5}. For each candidate,
# rebuild F for every feature and participant in both datasets, pool
# all rows, and compute per dataset:
#   F_IQR       = interquartile range of pooled F  (higher = less
#                 saturated / more graded)
#   |Sp(T,F)|   = |Spearman(T, F)| on pooled T/F
# Choose the SCALE that MAXIMISES the mean F_IQR across datasets,
# subject to the WORST-CASE (max across datasets) |Sp(T,F)| staying
# < 0.7. If no candidate satisfies the constraint on both datasets,
# keep the current CONFIG["F_SCALE"] and flag for manual review
# rather than silently picking a value that fails the independence
# check on one dataset.
# ================================================================
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

SCALE_GRID = [1, 2, 3, 4, 5]
_original_f_scale = ndi.CONFIG["F_SCALE"]
sweep_rows = []

for scale in SCALE_GRID:
    ndi.CONFIG["F_SCALE"] = scale   # neutrosophic_decompose reads CONFIG at call time

    for dataset_name, participants, feat_names in [
        ("DEPRESJON", depresjon_participants, FEAT_NAMES),
        ("PMData",    pmdata_participants,    PMDATA_FEATURE_NAMES),
    ]:
        if not participants:
            continue
        T_pool, F_pool = [], []
        for p in participants:
            df = p["days_df"].copy().reset_index(drop=True)
            for col in feat_names:
                if col in df.columns:
                    df[col] = pd.to_numeric(df[col], errors="coerce")
                    df[col] = df[col].fillna(df[col].median())
                else:
                    df[col] = 0.0
            X = df[feat_names].values
            n = len(X)
            train_mask_full = np.ones(n, dtype=bool)  # same in-sample mode as Cell 4/6
            neutro_df = ndi.build_neutrosophic_features(
                X, feat_names, train_mask=train_mask_full)
            T_pool.append(neutro_df["Depression_T"].values)
            F_pool.append(neutro_df["Depression_F"].values)

        T_all = np.concatenate(T_pool)
        F_all = np.concatenate(F_pool)
        valid = ~(np.isnan(T_all) | np.isnan(F_all))
        T_all, F_all = T_all[valid], F_all[valid]

        f_iqr = float(np.percentile(F_all, 75) - np.percentile(F_all, 25))
        sp, _ = spearmanr(T_all, F_all)

        sweep_rows.append(dict(scale=scale, dataset=dataset_name,
                                F_IQR=f_iqr, abs_Sp_TF=abs(sp)))

sweep_df = pd.DataFrame(sweep_rows)
print("=" * 65)
print("F_SCALE PRE-SPECIFICATION SWEEP")
print("=" * 65)
print(sweep_df.to_string(index=False))

agg = (sweep_df.groupby("scale")
       .agg(mean_F_IQR=("F_IQR", "mean"),
            max_abs_Sp_TF=("abs_Sp_TF", "max"))
       .reset_index())
agg["passes_constraint"] = agg["max_abs_Sp_TF"] < 0.7
print()
print(agg.to_string(index=False))

qualifying = agg[agg["passes_constraint"]]
if len(qualifying) > 0:
    best_row = qualifying.loc[qualifying["mean_F_IQR"].idxmax()]
    CHOSEN_SCALE = float(best_row["scale"])
    print(f"\nCHOSEN F_SCALE = {CHOSEN_SCALE}  "
          f"(mean F_IQR={best_row['mean_F_IQR']:.3f}, "
          f"worst-case |Sp(T,F)|={best_row['max_abs_Sp_TF']:.3f} < 0.7)")
else:
    CHOSEN_SCALE = _original_f_scale
    print(f"\nNo SCALE in {SCALE_GRID} keeps |Sp(T,F)| < 0.7 on BOTH "
          f"datasets -- keeping current CONFIG['F_SCALE']="
          f"{_original_f_scale} and flagging for manual review "
          f"(widen the grid, or revisit the normaliser).")

# FREEZE. This is the only place F_SCALE should be set after this
# point. Chosen by the rule above, before any CV/SHAP/lambda result
# was inspected -- do not hand-tune this later after seeing rho.
ndi.CONFIG["F_SCALE"] = CHOSEN_SCALE
print(f"\nndi.CONFIG['F_SCALE'] frozen at {CHOSEN_SCALE}.")
if CHOSEN_SCALE != _original_f_scale:
    print("This differs from the value already baked into "
          "depresjon_all / pmdata_all (built in Cell 4 / Cell 6) -- "
          "RE-RUN those two cells now so Cell 7's verification "
          "reflects the frozen value. Per-fold CV cells (Component 2 "
          "onward) rebuild features from raw data on every call, so "
          "they pick up CONFIG['F_SCALE'] automatically.")
else:
    print("Matches the value already used in Cell 4 / Cell 6 -- no "
          "rebuild needed.")

In [ ]:
# ================================================================
# CELL 14.4  V2' — POPULATION VOLATILITY NORM + FOLD-WISE FRAME
# (V2' spec Sec 4) Population statistic is now the SPREAD OF PER-
# PERSON VOLATILITIES across donors, not raw values. Leakage rule
# unchanged: p never enters its own donor pool.
# ================================================================
def _compute_population_volatility_norm(participants, held_out_pid, feat_names, has_labels):
    donors = [q for q in participants if q["pid"] != held_out_pid]
    if has_labels:
        donors = [q for q in donors if q.get("group") == "control"]
    donor_ids = [q["pid"] for q in donors]

    assert held_out_pid not in donor_ids, \
        f"LEAKAGE: {held_out_pid} is in its own donor pool"

    donor_vols = []
    for q in donors:
        dfq = q["days_df"]
        vols = []
        for col in feat_names:
            v = pd.to_numeric(dfq[col], errors="coerce") if col in dfq.columns \
                else pd.Series(np.nan, index=dfq.index)
            vols.append(v.std())
        donor_vols.append(vols)
    donor_vols = np.array(donor_vols, dtype=float)  # (n_donors, d)

    pop_vol_median = np.nanmedian(donor_vols, axis=0)
    pop_vol_mad    = np.nanmedian(np.abs(donor_vols - pop_vol_median), axis=0) + 1e-8
    return pop_vol_median, pop_vol_mad, donor_ids


def build_v2prime_diagnostic_frame_foldwise(participants, feat_names, has_labels):
    """Fold-wise (not whole-series) -- V2's degenerate-frame lesson
    applies here too: F must vary within a participant to test
    independence against."""
    frames = []
    for p in participants:
        df_raw = p.get("days_df")
        if df_raw is None:
            continue
        df_raw = df_raw.copy().reset_index(drop=True)
        for col in feat_names:
            if col in df_raw.columns:
                df_raw[col] = pd.to_numeric(df_raw[col], errors="coerce")
                df_raw[col] = df_raw[col].fillna(df_raw[col].median())
            else:
                df_raw[col] = 0.0
        n = len(df_raw)
        if n < 15:
            continue
        X_raw = df_raw[feat_names].values
        t0 = min(10, max(5, n // 3))

        pop_vol_median, pop_vol_mad, _ = _compute_population_volatility_norm(
            participants, p["pid"], feat_names, has_labels)

        try:
            fold_info = ndi.blocked_tscv(
                X_raw, np.zeros(n), ndi.rf_factory, T0=t0, return_folds=True)
            folds = fold_info.get("folds", [])
        except Exception:
            continue

        rows = []
        for fold in folds:
            tr_idx, te_idx = fold
            tr_idx = np.array(tr_idx); te_idx = np.array(te_idx)
            if int(te_idx.min()) - int(tr_idx.max()) < 6:
                continue
            train_mask = np.zeros(n, dtype=bool)
            train_mask[tr_idx] = True

            neutro_fold = ndi.build_neutrosophic_features(
                X_raw, feat_names, train_mask=train_mask,
                f_mode="population_vol",
                pop_vol_median=pop_vol_median, pop_vol_mad=pop_vol_mad)
            rows.append(neutro_fold.iloc[te_idx])

        if not rows:
            continue
        pf = pd.concat(rows, ignore_index=True)
        pf["pid"]   = p["pid"]
        pf["group"] = p.get("group", "pmdata")
        frames.append(pf)

    return pd.concat(frames, ignore_index=True)

In [ ]:
# ================================================================
# CELL 14.10  ITEM 2a -- V3_BEST_LAMBDA (frozen; run completed fully
# before this session disconnected -- both datasets printed, no
# need to rerun the 12hr search)
#
#   PMData:    best_lambda=(0.20, 0.15, 0.65)  mean_rho=0.3834
#              original_mean_rho=0.3198  n_stable=13  Robust
#   DEPRESJON: best_lambda=(0.00, 1.00, 0.00)  mean_rho=0.6325
#              original_mean_rho=0.2962  n_stable=1
#              NOT robust -- corner solution, near-zero stability
#              radius. Not used downstream (V3_BEST_LAMBDA freezes
#              PMData's), kept here only for the write-up's caveat.
# ================================================================
V3_BEST_LAMBDA = (0.20, 0.15, 0.65)

pmdata_lambda_result = dict(
    best_lambda=V3_BEST_LAMBDA, best_mean_rho=0.3834,
    original_mean_rho=0.3198, n_stable=13,
    verdict="Robust — weights stable across neighbourhood")

depresjon_lambda_result = dict(
    best_lambda=(0.0, 1.0, 0.0), best_mean_rho=0.6325,
    original_mean_rho=0.2962, n_stable=1,
    verdict="Per-subject tuning recommended (small stability radius) — corner solution, not robust")

print(f"V3_BEST_LAMBDA (frozen from PMData primary calibration): {V3_BEST_LAMBDA}")

In [ ]:
# ================================================================
# CELL 14.11  ITEM 2b -- THE COMPARISON (fast, spec Sec 7 primary endpoint)
#
# SPEED FIX (see chat): compute_T_v3, compute_quality_signal, and
# compute_F_v3 all use genuinely expanding, train_mask-INDEPENDENT
# causal statistics (same precedent as F's volatility in V2'). Their
# output for a given participant never changes based on which fold
# is being evaluated -- but the original cell recomputed all three
# fresh inside the fold loop, every fold, redoing identical work
# n_folds times. Fixed: compute once per participant, outside the
# loop, slice per fold. V-signed's OWN features genuinely DO depend
# on train_mask (personal baseline from that fold's training days)
# and stay inside the loop, unchanged -- same models, same arms,
# same protocol, zero deviation from spec.
# ================================================================
from sklearn.linear_model import Ridge

def _zscore_ridge_predict(X_tr, y_tr, X_te):
    """Personalised z-score reference (Brief Eq. 5) -- matches
    _RidgeModel exactly: causal per-fold z-score, then Ridge."""
    mu = X_tr.mean(0); sg = X_tr.std(0) + 1e-8
    m = Ridge(alpha=ndi.CONFIG["RIDGE_ALPHA"]).fit((X_tr - mu) / sg, y_tr)
    return m.predict((X_te - mu) / sg)

def run_v3_comparison_fast(participants, dataset_label, has_labels, feat_names, v3_lambda):
    print(f"\n--- {dataset_label}: comparison (V-signed / V3 / raw-RF / personalised-Z) ---")
    print(f"  {'PID':<14} {'n_folds':>8} {'Vsigned_rho':>12} {'V3_rho':>9} "
          f"{'RawRF_rho':>10} {'PersZ_rho':>10}")
    print("  " + "-" * 68)

    rows = []
    for p in participants:
        df_raw_orig = p.get("days_df")
        if df_raw_orig is None:
            continue
        df_raw_orig = df_raw_orig.copy().reset_index(drop=True)
        df_raw = df_raw_orig.copy()
        for col in feat_names:
            if col in df_raw.columns:
                df_raw[col] = pd.to_numeric(df_raw[col], errors="coerce")
                df_raw[col] = df_raw[col].fillna(df_raw[col].median())
            else:
                df_raw[col] = 0.0
        n = len(df_raw)
        if n < 15:
            continue
        X_raw = df_raw[feat_names].values
        t0 = min(10, max(5, n // 3))

        pop_vol_median, pop_vol_mad, _ = _compute_population_volatility_norm(
            participants, p["pid"], feat_names, has_labels)

        # ---- V3's T/I/F computed ONCE per participant (train_mask-independent) ----
        full_mask = np.ones(n, dtype=bool)
        T_v3_full = ndi.compute_T_v3(X_raw, full_mask)
        missing_full, instab_full = ndi.compute_quality_signal(df_raw_orig, feat_names, full_mask)
        I_v3_full = ndi.compute_I_v3(missing_full, instab_full)
        F_v3_full = ndi.compute_F_v3(X_raw, feat_names, full_mask, pop_vol_median, pop_vol_mad)
        X_v3_full = np.column_stack([T_v3_full, I_v3_full, F_v3_full])

        ndi_full = ndi.compute_ndi_v3_primary(T_v3_full, I_v3_full, F_v3_full, lambda_vec=(0.2, 0.3, 0.5))
        y_smooth = pd.Series(ndi_full).rolling(7, min_periods=1).mean().shift(1).fillna(0).values

        try:
            fold_info = ndi.blocked_tscv(X_raw, np.zeros(n), ndi.rf_factory,
                                          T0=t0, return_folds=True)
            folds = fold_info.get("folds", [])
        except Exception:
            continue
        if not folds:
            continue

        preds = {"vsigned": [], "v3": [], "rawrf": [], "persz": []}
        y_true_all = []

        for tr_idx, te_idx in folds:
            tr_idx, te_idx = np.array(tr_idx), np.array(te_idx)
            if int(te_idx.min()) - int(tr_idx.max()) < 6:
                continue
            train_mask = np.zeros(n, dtype=bool)
            train_mask[tr_idx] = True

            # V-signed: genuinely fold-dependent -- rebuild, unchanged.
            vs_neutro = ndi.build_neutrosophic_features(X_raw, feat_names, train_mask=train_mask)
            X_vs = vs_neutro[["Depression_T", "Depression_I", "Depression_F"]].fillna(0).values

            y_fold = y_smooth
            tr_valid = tr_idx[np.isfinite(y_fold[tr_idx])]
            te_valid = te_idx[np.isfinite(y_fold[te_idx])]
            if len(tr_valid) < 5 or len(te_valid) < 1:
                continue

            try:
                rf_vs = ndi.rf_factory(); rf_vs.fit(X_vs[tr_valid], y_fold[tr_valid])
                pred_vs = rf_vs.predict(X_vs[te_valid])

                # V3: just slice the precomputed, fold-independent features.
                rf_v3 = ndi.rf_factory(); rf_v3.fit(X_v3_full[tr_valid], y_fold[tr_valid])
                pred_v3 = rf_v3.predict(X_v3_full[te_valid])

                rf_raw = ndi.rf_factory(); rf_raw.fit(X_raw[tr_valid], y_fold[tr_valid])
                pred_raw = rf_raw.predict(X_raw[te_valid])

                pred_z = _zscore_ridge_predict(X_raw[tr_valid], y_fold[tr_valid], X_raw[te_valid])
            except Exception:
                continue

            preds["vsigned"].extend(pred_vs.tolist())
            preds["v3"].extend(pred_v3.tolist())
            preds["rawrf"].extend(pred_raw.tolist())
            preds["persz"].extend(pred_z.tolist())
            y_true_all.extend(y_fold[te_valid].tolist())

        if len(y_true_all) < 3:
            continue

        y_true_arr = np.array(y_true_all)
        rhos = {}
        for arm in ["vsigned", "v3", "rawrf", "persz"]:
            pred_arr = np.array(preds[arm])
            mask = np.isfinite(y_true_arr) & np.isfinite(pred_arr)
            if mask.sum() < 3:
                rhos[arm] = np.nan
                continue
            r, _ = spearmanr(y_true_arr[mask], pred_arr[mask])
            rhos[arm] = r

        n_folds = len(y_true_all)
        rows.append(dict(pid=p["pid"], n_folds=n_folds, **rhos))
        print(f"  {p['pid']:<14} {n_folds:>8} {rhos['vsigned']:>12.4f} {rhos['v3']:>9.4f} "
              f"{rhos['rawrf']:>10.4f} {rhos['persz']:>10.4f}")

    return pd.DataFrame(rows)


pmdata_comparison = run_v3_comparison_fast(
    pmdata_participants, "PMData", has_labels=False,
    feat_names=PMDATA_FEATURE_NAMES, v3_lambda=V3_BEST_LAMBDA)

pmdata_primary = pmdata_comparison[pmdata_comparison["n_folds"] >= 10]

print("\n" + "="*65); print("PRIMARY ENDPOINT -- PMData, n_folds>=10, all 4 arms"); print("="*65)
for arm, label in [("vsigned","V-signed"), ("v3","V3"),
                    ("rawrf","Raw-feature RF"), ("persz","Personalised Z")]:
    vals = pmdata_primary[arm].dropna()
    if len(vals) > 0:
        print(f"  {label:<18} n={len(vals):>3}  mean rho={vals.mean():.4f}  "
              f"SD={vals.std():.4f}  median={vals.median():.4f}")
    else:
        print(f"  {label:<18} n=0 (no eligible participants)")

depresjon_comparison = run_v3_comparison_fast(
    depresjon_participants, "DEPRESJON", has_labels=True,
    feat_names=FEAT_NAMES, v3_lambda=V3_BEST_LAMBDA)

print("\n" + "="*65); print("SECONDARY -- DEPRESJON, all participants, all 4 arms"); print("="*65)
for arm, label in [("vsigned","V-signed"), ("v3","V3"),
                    ("rawrf","Raw-feature RF"), ("persz","Personalised Z")]:
    vals = depresjon_comparison[arm].dropna()
    if len(vals) > 0:
        print(f"  {label:<18} n={len(vals):>3}  mean rho={vals.mean():.4f}  SD={vals.std():.4f}")
save_result(3, "Primary: within-participant rho, PMData n_folds>=10, all 4 arms",
            pmdata_primary, "primary_pmdata_rho.csv", "CELL 14.11")
save_result(3, "Secondary: DEPRESJON, all participants, all 4 arms (context, not primary)",
            depresjon_comparison, "secondary_depresjon_rho.csv", "CELL 14.11")

In [ ]:
# ================================================================
# 14.11 R1-a RUN 1 (PRIORITY) — RAW-FEATURE TARGET, DE-RISKS THE HEADLINE CLAIM
# Repeats the exact primary comparison, same protocol, same 15
# PMData participants, same folds — but predicts a target that is
# NOT neutrosophic-derived, so the q=0.030 advantage can't be
# "features that built the target predict the target."
# ================================================================
from scipy.stats import zscore, wilcoxon

def run_v3_comparison_raw_target(participants, dataset_label, has_labels, feat_names):
    print(f"\n--- {dataset_label}: RAW-FEATURE TARGET comparison (V-signed / V3 / raw-RF / personalised-Z) ---")
    print(f"  {'PID':<14} {'n_folds':>8} {'Vsigned_rho':>12} {'V3_rho':>9} "
          f"{'RawRF_rho':>10} {'PersZ_rho':>10}")
    print("  " + "-" * 68)

    rows = []
    for p in participants:
        df_raw_orig = p.get("days_df")
        if df_raw_orig is None:
            continue
        df_raw_orig = df_raw_orig.copy().reset_index(drop=True)
        df_raw = df_raw_orig.copy()
        for col in feat_names:
            if col in df_raw.columns:
                df_raw[col] = pd.to_numeric(df_raw[col], errors="coerce")
                df_raw[col] = df_raw[col].fillna(df_raw[col].median())
            else:
                df_raw[col] = 0.0
        n = len(df_raw)
        if n < 15:
            continue
        X_raw = df_raw[feat_names].values
        t0 = min(10, max(5, n // 3))

        pop_vol_median, pop_vol_mad, _ = _compute_population_volatility_norm(
            participants, p["pid"], feat_names, has_labels)

        # ---- V3's T/I/F computed ONCE per participant (train_mask-independent) ----
        full_mask = np.ones(n, dtype=bool)
        T_v3_full = ndi.compute_T_v3(X_raw, full_mask)
        missing_full, instab_full = ndi.compute_quality_signal(df_raw_orig, feat_names, full_mask)
        I_v3_full = ndi.compute_I_v3(missing_full, instab_full)
        F_v3_full = ndi.compute_F_v3(X_raw, feat_names, full_mask, pop_vol_median, pop_vol_mad)
        X_v3_full = np.column_stack([T_v3_full, I_v3_full, F_v3_full])

        # ---- Target = next-day value of a FIXED raw-feature composite (no NDI in it) ----
        # z-scored mean of the raw feature matrix, lag-1, same 7-day smoothing:
        mu_raw = X_raw.mean(axis=0)
        sd_raw = X_raw.std(axis=0) + 1e-8
        raw_composite = ((X_raw - mu_raw) / sd_raw).mean(axis=1)
        y_target = pd.Series(raw_composite).rolling(7, min_periods=1).mean().shift(1)
        y_fold_full = y_target.values
        # all four arms predict THIS

        try:
            fold_info = ndi.blocked_tscv(X_raw, np.zeros(n), ndi.rf_factory,
                                          T0=t0, return_folds=True)
            folds = fold_info.get("folds", [])
        except Exception:
            continue
        if not folds:
            continue

        preds = {"vsigned": [], "v3": [], "rawrf": [], "persz": []}
        y_true_all = []

        for tr_idx, te_idx in folds:
            tr_idx, te_idx = np.array(tr_idx), np.array(te_idx)
            if int(te_idx.min()) - int(tr_idx.max()) < 6:
                continue
            train_mask = np.zeros(n, dtype=bool)
            train_mask[tr_idx] = True

            # V-signed: genuinely fold-dependent -- rebuild, unchanged.
            vs_neutro = ndi.build_neutrosophic_features(X_raw, feat_names, train_mask=train_mask)
            X_vs = vs_neutro[["Depression_T", "Depression_I", "Depression_F"]].fillna(0).values

            y_fold = y_fold_full
            tr_valid = tr_idx[np.isfinite(y_fold[tr_idx])]
            te_valid = te_idx[np.isfinite(y_fold[te_idx])]
            if len(tr_valid) < 5 or len(te_valid) < 1:
                continue

            try:
                rf_vs = ndi.rf_factory(); rf_vs.fit(X_vs[tr_valid], y_fold[tr_valid])
                pred_vs = rf_vs.predict(X_vs[te_valid])

                rf_v3 = ndi.rf_factory(); rf_v3.fit(X_v3_full[tr_valid], y_fold[tr_valid])
                pred_v3 = rf_v3.predict(X_v3_full[te_valid])

                rf_raw = ndi.rf_factory(); rf_raw.fit(X_raw[tr_valid], y_fold[tr_valid])
                pred_raw = rf_raw.predict(X_raw[te_valid])

                pred_z = _zscore_ridge_predict(X_raw[tr_valid], y_fold[tr_valid], X_raw[te_valid])
            except Exception:
                continue

            preds["vsigned"].extend(pred_vs.tolist())
            preds["v3"].extend(pred_v3.tolist())
            preds["rawrf"].extend(pred_raw.tolist())
            preds["persz"].extend(pred_z.tolist())
            y_true_all.extend(y_fold[te_valid].tolist())

        if len(y_true_all) < 3:
            continue

        y_true_arr = np.array(y_true_all)
        rhos = {}
        for arm in ["vsigned", "v3", "rawrf", "persz"]:
            pred_arr = np.array(preds[arm])
            mask = np.isfinite(y_true_arr) & np.isfinite(pred_arr)
            if mask.sum() < 3:
                rhos[arm] = np.nan
                continue
            r, _ = spearmanr(y_true_arr[mask], pred_arr[mask])
            rhos[arm] = r

        n_folds = len(y_true_all)
        rows.append(dict(pid=p["pid"], n_folds=n_folds, **rhos))
        print(f"  {p['pid']:<14} {n_folds:>8} {rhos['vsigned']:>12.4f} {rhos['v3']:>9.4f} "
              f"{rhos['rawrf']:>10.4f} {rhos['persz']:>10.4f}")

    return pd.DataFrame(rows)


pmdata_raw_target = run_v3_comparison_raw_target(
    pmdata_participants, "PMData", has_labels=False,
    feat_names=PMDATA_FEATURE_NAMES)

pmdata_raw_target_primary = pmdata_raw_target[pmdata_raw_target["n_folds"] >= 10]

print("\n" + "="*65); print("RUN 1 -- PRIMARY ENDPOINT ON RAW-FEATURE TARGET, PMData, n_folds>=10"); print("="*65)
for arm, label in [("vsigned","V-signed"), ("v3","V3"),
                    ("rawrf","Raw-feature RF"), ("persz","Personalised Z")]:
    vals = pmdata_raw_target_primary[arm].dropna()
    if len(vals) > 0:
        print(f"  {label:<18} n={len(vals):>3}  mean rho={vals.mean():.4f}  "
              f"SD={vals.std():.4f}  median={vals.median():.4f}")

In [ ]:
# ================================================================
# 14.11 R1-b RUN 1 -- REPEAT THE FOUR PAIRED WILCOXON TESTS + BH, RAW TARGET
# Pre-state the two outcomes so the interpretation is fixed before
# the number is seen. Both sentences are honest; only one is correct.
# ================================================================
paired_comparisons = [
    ("v3",      "rawrf", "V3 vs raw-RF"),
    ("v3",      "persz", "V3 vs pers-Z"),
    ("vsigned", "rawrf", "V-signed vs raw-RF"),
    ("vsigned", "persz", "V-signed vs pers-Z"),
]

run1_rows = []
df = pmdata_raw_target_primary
for arm_a, arm_b, label in paired_comparisons:
    d = df[[arm_a, arm_b]].dropna()
    diff = d[arm_a] - d[arm_b]
    try:
        stat, pv = wilcoxon(d[arm_a], d[arm_b])
    except ValueError:
        pv = np.nan
    n_wins = int((diff > 0).sum())
    run1_rows.append(dict(comparison=label, n=len(d), n_wins=n_wins, p=pv))

run1_df = pd.DataFrame(run1_rows)
run1_df["q"] = benjamini_hochberg(run1_df["p"].values)

print("="*65)
print("RUN 1 -- RAW-FEATURE TARGET, FOUR PAIRED WILCOXON TESTS + BH")
print("="*65)
print(run1_df.to_string(index=False))

# ---- pre-stated interpretation (decided by the numbers, not before) ----
all_significant = (run1_df["q"] < 0.05).all()
print("\n" + "="*65)
if all_significant:
    print("RUN 1 CONFIRMS: both neutrosophic arms still beat the references")
    print("on the raw target. Advantage is representation-level, not target")
    print("affinity. Headline claim is bulletproof.\n")
    print("Under leakage-free blocked cross-validation, both neutrosophic "
          "formulations significantly outperformed a raw-feature random "
          "forest and a personalised z-score baseline (all four paired "
          "comparisons q < 0.05, Benjamini-Hochberg), and this advantage "
          "held whether the prediction target was the neutrosophic "
          "composite or a raw-feature composite. The two neutrosophic "
          "formulations did not differ from each other (p = 0.56), "
          "indicating the advantage arises from the neutrosophic "
          "representation rather than the specific falsity construction.")
else:
    print("RUN 1 DOES NOT CONFIRM: advantage shrinks to non-significant on")
    print("the raw target. The earlier gap was target affinity.\n")
    print("The apparent advantage of the neutrosophic representation over "
          "naive baselines was present only when predicting a "
          "neutrosophic-derived target and did not survive on a "
          "raw-feature target (q > 0.05), indicating it reflected shared "
          "structure between the representation and the target rather "
          "than superior predictive content.")

save_result(4, "Raw-target rerun: 4 paired Wilcoxon + BH q (decisive null)",
            run1_df, "raw_target_wilcoxon.csv", "CELL 14.11 R1-b")

## v1.4: repository, config and provenance

In [ ]:
# G-0 (v1.4) Run from inside the repository so the provenance cell can see git (v1.4 memo, Section 4)
# Works with the private repository: if a plain clone/fetch fails it asks once for a read-only token
# (typed into a hidden prompt, never stored in the notebook or in .git/config), then checks out the
# tag so the run's HEAD is the tagged commit. Idempotent.
import os, subprocess, getpass
REPO_URL = "https://github.com/hpatel09112004/NDI_Depression_Risk_Index.git"
REPO_DIR = "/content/NDI_Depression_Risk_Index"
RUN_TAG = "v1.4"
_env = dict(os.environ, GIT_TERMINAL_PROMPT="0")

def _git_run(args, secret=None):
    r = subprocess.run(args, capture_output=True, text=True, env=_env)
    msg = (r.stderr or "")[-600:]
    return r.returncode, (msg.replace(secret, "***") if secret else msg)

def _token_url():
    tok = getpass.getpass("Private repository: paste the read-only GitHub token (hidden, not saved): ").strip()
    return REPO_URL.replace("https://", f"https://hpatel09112004:{tok}@"), tok

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    _rc, _msg = _git_run(["git", "clone", "--quiet", REPO_URL, REPO_DIR])
    if _rc != 0:
        _url, _tok = _token_url()
        _rc, _msg = _git_run(["git", "clone", "--quiet", _url, REPO_DIR], secret=_tok)
        assert _rc == 0, "git clone failed (token or URL?):\n" + _msg
        _git_run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL])   # token not kept on disk
else:
    _rc, _msg = _git_run(["git", "-C", REPO_DIR, "fetch", "--tags", "--quiet"])
    _have = _git_run(["git", "-C", REPO_DIR, "rev-parse", "-q", "--verify", f"refs/tags/{RUN_TAG}"])[0] == 0
    if _rc != 0 and not _have:
        _url, _tok = _token_url()
        _rc, _msg = _git_run(["git", "-C", REPO_DIR, "fetch", "--tags", "--quiet", _url], secret=_tok)
        assert _rc == 0, "git fetch failed (token or URL?):\n" + _msg
os.chdir(REPO_DIR)
if _git_run(["git", "rev-parse", "-q", "--verify", f"refs/tags/{RUN_TAG}"])[0] == 0:
    _rc, _msg = _git_run(["git", "checkout", "--quiet", "--force", RUN_TAG])   # --force: a config file left by an untagged attempt is replaced by the tagged one
    assert _rc == 0, f"could not check out {RUN_TAG}:\n" + _msg
    print(f"Checked out tag {RUN_TAG} (detached HEAD at the tagged commit)")
else:
    print(f"Tag {RUN_TAG} not found yet: V12-0 will write the config and stop.")
print("Working directory:", os.getcwd())
print("HEAD:", subprocess.run(["git", "log", "-1", "--format=%H %cI %s"], capture_output=True, text=True).stdout.strip())
print("Tags:", subprocess.run(["git", "tag", "--list"], capture_output=True, text=True).stdout.split())

In [ ]:
# V12-0 (v1.4) Imports, required globals, config, provenance
# Changed from v1.3: VERSION, tag v1.4, config_v1_4.json, output folder results_v1.4, results_v1.3 kept read-only,
# config keys for the data rows, frozen F_SCALE, p12 decision and wellness rules W1-W4 (v1.4 memo, Sections 2-4).
# Every analysis parameter is unchanged from v1.3.
import os, json, time, subprocess, datetime, warnings
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, wilcoxon, bootstrap as scipy_bootstrap
from sklearn.linear_model import Ridge

_REQUIRED = ["ndi", "pmdata_participants", "PMDATA_FEATURE_NAMES", "V3_BEST_LAMBDA",
             "RANDOM_SEED", "RESULTS_DIR", "_compute_population_volatility_norm",
             "pmdata_primary", "pmdata_raw_target_primary", "DATE_CHECK", "wellness_log"]
_missing = [g for g in _REQUIRED if g not in globals()]
assert not _missing, f"Run the notebook up to CELL 14.11 R1-b first. Missing: {_missing}"

V1_2_CONFIG = dict(                  # variable name kept from v1.2 so the cells below run unchanged
    VERSION="v1.4",
    FEATURE_NORMALISATION="fold_local_truncated_at_test_day",
    IMPUTATION_FEATURES="training_fold_median",
    IMPUTATION_TARGETS="causal_expanding_median",
    TARGET_STATS="causal_expanding",
    TARGET_BURN_IN_DAYS=14,          # capped per participant so burn-in is training-only in every fold
    TARGET_ALIGNMENT="forward",      # v1.3 decision: y[s] = mean index over rows s+1 .. s+7
    FORWARD_TARGET_MIN_PERIODS=7,    # full 7-row window required; shorter window -> NaN -> test day dropped
    PURGE_SUPPORT_OVERLAP=True,
    SMOOTH_K=7, EMBARGO_G=6, MIN_FOLDS=10,
    FIXED_LAMBDA=[0.2, 0.3, 0.5],
    LAMBDA_STAR=[float(x) for x in V3_BEST_LAMBDA],
    SESOI=0.10, BOOTSTRAP_N=10000, BOOTSTRAP_METHOD="BCa", BOOTSTRAP_SEED=int(RANDOM_SEED),
    LEAK_DEFINITIONS=dict(
        L0="embargo off (g=0) and no purge; literature-known reference leak",
        L1="whole-series normalisation: T/I/F, V-signed and imputation from the full series (v1 feature path)",
        L2="self-predictive proxy: target index at the most recent day, no model",
        L3="in-sample weight selection: no pathway into RF-on-[T,I,F] (lambda never enters features) -- inert",
        L4="target index built with tuned lambda* instead of fixed (0.2,0.3,0.5)"),
    LEAK_REMOVAL_ORDER=["L2", "L4", "L3", "L1", "L0"],
    INERT_LEAKS=["L3"],
    SENSITIVITY_ROWS=[],             # per-fold target rebuild is not applicable under forward alignment
    EXPLORATORY=["target-swap shift: (delta on self-derived target) - (delta on raw-feature target), "
                 "four comparisons, mean, BCa 95% CI, Wilcoxon P, BH q"],
    REPORTING_DECIMALS=3,
    # ---- v1.4 additions (v1.4 memo) ----
    DATA_ROWS="one row per local calendar date, built from the Fitbit daily files; wellness is never merged into "
              "days_df; dates asserted unique and consecutive for every participant (cell D-2)",
    F_SCALE_FROZEN=1.0,              # chosen by the CELL 6.5 rule in v1; asserted below, not re-chosen
    P12_DECISION="max heart rate read with pd.to_numeric(errors='coerce') (cell held a single space; max HR is not "
                 "a model feature). p12 enters only through the existing rules (>= 15 rows, >= 10 scored test "
                 "folds). If it enters, the primary numbers are also reported without p12, with p12 removed from "
                 "the cohort and from every donor pool",
    WELLNESS_RULES=dict(
        LOCAL_TZ="Europe/Oslo",
        ITEMS_1_TO_5=["mood", "stress", "fatigue", "sleep_quality", "soreness"],
        W1="date each report by its local calendar day (Europe/Oslo), from the tz-aware timestamp",
        W2="for the five items scored 1-5, 0 = missing; readiness runs 0-10, so 0 is valid there",
        W3="a report whose five 1-5 items are all 0 is a blank submission: drop the whole report, before W4",
        W4="several reports on one local day: keep the last valid report of that day, item by item after W2; "
           "no valid value for an item that day -> missing"),
    WELLNESS_USE="separate per-participant table, one row per local date (cell W-1); joined only when the Task 3 "
                 "target is built; not used in Tasks 1-2",
)

TAG = "v1.4"
CONFIG_NAME = "config_v1_4.json"
_BASE_DIR = os.path.dirname(RESULTS_DIR)
RESULTS_V1_4_DIR = os.path.join(_BASE_DIR, "results_v1.4")
RESULTS_V1_3_ARCHIVE = os.path.join(_BASE_DIR, "results_v1.3")   # read-only from here on
RESULTS_V1_2_ARCHIVE = os.path.join(_BASE_DIR, "results_v1.2")   # read-only from here on
RESULTS_V1_2_DIR = RESULTS_V1_4_DIR  # legacy variable name used by V12-5 .. V12-8: ALL output goes to results_v1.4
assert os.path.basename(RESULTS_V1_2_DIR) == "results_v1.4"
_ARCHIVE_NEEDS = {RESULTS_V1_3_ARCHIVE: ["table1_corrected_self_target_per_participant.csv", "raw_target_corrected_per_participant.csv",
                                         "three_way_v1_v1_2_v1_3.csv", "three_way_v1_v1_2_v1_3_formatted.csv",
                                         "task2_one_at_a_time.csv", "task2_waterfall_cumulative.csv",
                                         "forward_target_dropped_test_days.csv"],
                  RESULTS_V1_2_ARCHIVE: ["task2_one_at_a_time.csv", "task2_waterfall_cumulative.csv"]}
_absent = [os.path.join(d, f) for d, fs in _ARCHIVE_NEEDS.items() for f in fs if not os.path.isfile(os.path.join(d, f))]
assert not _absent, f"archived files needed by X-1/X-5 are missing: {_absent}"
assert float(ndi.CONFIG["F_SCALE"]) == V1_2_CONFIG["F_SCALE_FROZEN"], \
    f"CELL 6.5 chose F_SCALE = {ndi.CONFIG['F_SCALE']} on the v1.4 data, not 1.0 -- STOP and report to ma'am"
os.makedirs(RESULTS_V1_4_DIR, exist_ok=True)
CONFIG_PATH = os.path.join(RESULTS_V1_4_DIR, CONFIG_NAME)
with open(CONFIG_PATH, "w") as f:
    json.dump(V1_2_CONFIG, f, indent=2)
with open(CONFIG_NAME, "w") as f:    # repo copy -- this is the file that is committed and tagged
    json.dump(V1_2_CONFIG, f, indent=2)

def _git(*args):
    try:
        return subprocess.check_output(["git", *args], stderr=subprocess.DEVNULL, text=True).strip()
    except Exception:
        return None

_now = datetime.datetime.now(datetime.timezone.utc)
_tag_commit = _git("rev-list", "-n", "1", TAG)
_tag_time = _git("log", "-1", "--format=%cI", TAG)
_tagged_cfg = _git("show", f"{TAG}:{CONFIG_NAME}")       # the config as committed under the tag
try:
    _cfg_same = _tagged_cfg is not None and json.loads(_tagged_cfg) == json.loads(json.dumps(V1_2_CONFIG))
except Exception:
    _cfg_same = False
PROVENANCE = dict(version=V1_2_CONFIG["VERSION"],
                  run_started=_now.isoformat(timespec="seconds"),
                  working_directory=os.getcwd(),
                  head_commit=_git("rev-parse", "HEAD"),
                  tag=TAG, tag_commit=_tag_commit, tag_commit_time=_tag_time,
                  config_matches_tagged_config=bool(_cfg_same),
                  tag_time_precedes_run_start=bool(_tag_time and datetime.datetime.fromisoformat(_tag_time) < _now),
                  pmdata_participants_loaded=[p["pid"] for p in pmdata_participants],
                  f_scale=float(ndi.CONFIG["F_SCALE"]))
print(json.dumps(PROVENANCE, indent=2))

ALLOW_UNTAGGED_RUN = False           # leave False
_tag_ok = bool(_tag_commit) and _cfg_same and PROVENANCE["tag_time_precedes_run_start"]
if not _tag_ok and not ALLOW_UNTAGGED_RUN:
    raise RuntimeError(
        f"STOP: this run is not tagged. {CONFIG_NAME} has been written to {os.getcwd()}.\n"
        f"Commit it together with this notebook, run `git tag {TAG}`, `git push origin HEAD --tags`,\n"
        "then Runtime -> Restart session and Run all. (tag found: %s | config matches tag: %s | tag precedes run: %s)"
        % (bool(_tag_commit), _cfg_same, PROVENANCE["tag_time_precedes_run_start"]))
with open(os.path.join(RESULTS_V1_4_DIR, "provenance.json"), "w") as f:
    json.dump(PROVENANCE, f, indent=2)
print(f"TAG OK: {TAG} -> {_tag_commit[:10] if _tag_commit else None}, committed {_tag_time}; outputs -> {RESULTS_V1_4_DIR}")

SEED, N_BOOT, SESOI = V1_2_CONFIG["BOOTSTRAP_SEED"], V1_2_CONFIG["BOOTSTRAP_N"], V1_2_CONFIG["SESOI"]
K, G = V1_2_CONFIG["SMOOTH_K"], V1_2_CONFIG["EMBARGO_G"]
ARMS = ["vsigned", "v3", "rawrf", "persz"]
ARM_LABEL = {"vsigned": "V-signed", "v3": "V3", "rawrf": "Raw-feature RF", "persz": "Personalised Z"}
COMPARISONS = [("v3", "rawrf", "V3 vs raw-RF"), ("v3", "persz", "V3 vs pers-Z"),
               ("vsigned", "rawrf", "V-signed vs raw-RF"), ("vsigned", "persz", "V-signed vs pers-Z")]

## Corrected path (runner unchanged from v1.3)

In [ ]:
# V12-1 (v1.3: forward branch of smooth_target uses min_periods=k) Building blocks: imputation, causal targets, target support, purge
def _feature_matrix_nan(d0, feat_names):
    """Raw feature matrix with NaN kept (columns absent from the frame -> 0.0, as in v1)."""
    n = len(d0)
    cols = [pd.to_numeric(d0[c], errors="coerce").values if c in d0.columns else np.zeros(n)
            for c in feat_names]
    return np.column_stack(cols).astype(float)

def impute_from_rows(X_nan, row_mask):
    """Fill NaN with the column median over row_mask only (column all-NaN there -> 0.0)."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        med = np.nanmedian(X_nan[row_mask], axis=0)
    med = np.where(np.isfinite(med), med, 0.0)
    return np.where(np.isnan(X_nan), med, X_nan)

def impute_whole_series_v1(d0, feat_names):
    """Exact v1 behaviour: whole-series median per column."""
    df = d0.copy()
    for col in feat_names:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = 0.0
    return df[feat_names].values

def impute_causal(X_nan, B):
    """Row d filled with the median of rows [0, max(d+1, B)). Causal after the burn-in."""
    n = len(X_nan); out = X_nan.copy()
    for d in range(n):
        if np.isnan(X_nan[d]).any():
            L = min(max(d + 1, B), n)
            m = np.zeros(n, bool); m[:L] = True
            out[d] = impute_from_rows(X_nan, m)[d]
    return out

def smooth_target(idx, alignment, k=K):
    s = pd.Series(np.asarray(idx, float))
    if alignment == "trailing_v1":      # y[s] = mean(idx[s-k .. s-1])  (v1 convention)
        return s.rolling(k, min_periods=1).mean().shift(1).values
    if alignment == "forward":          # y[s] = mean(idx[s+1 .. s+k])
        # v1.3: full window required (min_periods=k). Rows with fewer than k following rows get NaN
        # and are dropped from scoring; with min_periods=1 they averaged fewer than 7 days.
        return s[::-1].rolling(k, min_periods=k).mean()[::-1].shift(-1).values
    raise ValueError(alignment)

def target_support(s, alignment, n, k=K):
    """Days whose index values enter y[s]."""
    if alignment == "trailing_v1":
        return set(range(max(0, s - k), s))
    return set(range(s + 1, min(n, s + k + 1)))

def purge_training_rows(tr_idx, te_idx, alignment, n):
    """Drop training rows whose target window touches the test day or its target window,
    or that lie inside the test day's target window."""
    test_days = set(int(t) for t in te_idx)
    test_support = set().union(*[target_support(int(s), alignment, n) for s in te_idx])
    blocked = test_days | test_support
    keep = [int(t) for t in tr_idx
            if int(t) not in test_support and not (target_support(int(t), alignment, n) & blocked)]
    return np.array(keep, dtype=int), len(tr_idx) - len(keep)


In [ ]:
# V12-2 Participant preparation, causal channels, the unified runner
_PREP_CACHE, _CAUSAL_CACHE, _RUN_CACHE = {}, {}, {}
RUN_LOGS = {}   # label -> per-participant fold log (burn-in used, rows purged, first fold days)

def prepare(p, feat_names, has_labels, participants):
    key = (p["pid"], tuple(feat_names), bool(has_labels))
    if key in _PREP_CACHE:
        return _PREP_CACHE[key]
    d0 = p["days_df"].copy().reset_index(drop=True)
    n = len(d0)
    X_nan = _feature_matrix_nan(d0, feat_names)
    X_ws = impute_whole_series_v1(d0, feat_names)
    pm, pmad, _ = _compute_population_volatility_norm(participants, p["pid"], feat_names, has_labels)
    full = np.ones(n, bool)
    T_ws = ndi.compute_T_v3(X_ws, full)
    mi, ii = ndi.compute_quality_signal(d0, feat_names, full)
    I_ws = ndi.compute_I_v3(mi, ii)
    F_ws = ndi.compute_F_v3(X_ws, feat_names, full, pm, pmad)
    out = dict(pid=p["pid"], d0=d0, n=n, X_nan=X_nan, X_ws=X_ws, pm=pm, pmad=pmad,
               T_ws=T_ws, I_ws=I_ws, F_ws=F_ws, feat_names=list(feat_names),
               t0=min(10, max(5, n // 3)))
    _PREP_CACHE[key] = out
    return out

def get_folds(pr, g):
    fi = ndi.blocked_tscv(pr["X_ws"], np.zeros(pr["n"]), ndi.rf_factory,
                          T0=pr["t0"], g=g, return_folds=True)
    return [(np.array(a, int), np.array(b, int)) for a, b in fi.get("folds", [])]

def causal_channels(pr, B):
    """T, I, F where day d is computed from rows [0, max(d+1, B)) only."""
    key = (pr["pid"], tuple(pr["feat_names"]), B)
    if key in _CAUSAL_CACHE:
        return _CAUSAL_CACHE[key]
    n, fn = pr["n"], pr["feat_names"]
    Xc = impute_causal(pr["X_nan"], B)
    T, I, F = (np.full(n, np.nan) for _ in range(3))
    for d in range(n):
        L = min(max(d + 1, B), n); ones = np.ones(L, bool)
        T[d] = ndi.compute_T_v3(Xc[:L], ones)[d]
        F[d] = ndi.compute_F_v3(Xc[:L], fn, ones, pr["pm"], pr["pmad"])[d]
        mi, ii = ndi.compute_quality_signal(pr["d0"].iloc[:L].reset_index(drop=True), fn, ones)
        I[d] = ndi.compute_I_v3(mi, ii)[d]
    _CAUSAL_CACHE[key] = (T, I, F, Xc)
    return _CAUSAL_CACHE[key]

def build_target(pr, target, target_stats, alignment, lam, B):
    """Returns (y, index_series_or_None)."""
    n = pr["n"]
    if target_stats == "whole_series_v1":
        if target == "self":
            idx = ndi.compute_ndi_v3_primary(pr["T_ws"], pr["I_ws"], pr["F_ws"], lambda_vec=tuple(lam))
            y = pd.Series(idx).rolling(K, min_periods=1).mean().shift(1).fillna(0).values
            return y, np.asarray(idx, float)
        X = pr["X_ws"]; comp = ((X - X.mean(0)) / (X.std(0) + 1e-8)).mean(1)
        return pd.Series(comp).rolling(K, min_periods=1).mean().shift(1).values, None
    T, I, F, Xc = causal_channels(pr, B)
    if target == "self":
        idx = np.asarray(ndi.compute_ndi_v3_primary(T, I, F, lambda_vec=tuple(lam)), float)
        return smooth_target(idx, alignment), idx
    comp = np.full(n, np.nan)
    for d in range(n):
        L = min(max(d + 1, B), n)
        mu, sd = Xc[:L].mean(0), Xc[:L].std(0) + 1e-8
        comp[d] = ((Xc[d] - mu) / sd).mean()
    return smooth_target(comp, alignment), None

def fold_features(pr, tr_idx, te_idx, whole_series, need_vsigned):
    fn = pr["feat_names"]
    if whole_series:                                  # v1 feature path (= leak L1)
        n = pr["n"]; tm = np.zeros(n, bool); tm[tr_idx] = True
        X = pr["X_ws"]
        X3 = np.column_stack([pr["T_ws"], pr["I_ws"], pr["F_ws"]])
    else:                                             # corrected: training stats, truncated at test day
        L = int(te_idx.max()) + 1
        tm = np.zeros(L, bool); tm[tr_idx] = True
        X = impute_from_rows(pr["X_nan"][:L], tm)
        T = ndi.compute_T_v3(X, tm)
        mi, ii = ndi.compute_quality_signal(pr["d0"].iloc[:L].reset_index(drop=True), fn, tm)
        I = ndi.compute_I_v3(mi, ii)
        F = ndi.compute_F_v3(X, fn, tm, pr["pm"], pr["pmad"])
        X3 = np.column_stack([T, I, F])
    XVS = None
    if need_vsigned:
        vs = ndi.build_neutrosophic_features(X, fn, train_mask=tm)
        XVS = vs[["Depression_T", "Depression_I", "Depression_F"]].fillna(0).values
    return X, X3, XVS

def _zscore_ridge(X_tr, y_tr, X_te):
    mu = X_tr.mean(0); sg = X_tr.std(0) + 1e-8
    return Ridge(alpha=ndi.CONFIG["RIDGE_ALPHA"]).fit((X_tr - mu) / sg, y_tr).predict((X_te - mu) / sg)

def run_arms(participants, feat_names, has_labels, target="self", arms=ARMS,
             target_stats="causal", alignment=None, whole_series_features=False,
             g=G, purge=None, lam=None, burn_in=None, target_mode="fixed", label=""):
    """One runner for Table 1, the raw-target comparison and every Task 2 state.
    target: "self" | "raw".  target_mode: "fixed" (primary) | "per_fold" (sensitivity row)."""
    alignment = alignment or V1_2_CONFIG["TARGET_ALIGNMENT"]
    purge = V1_2_CONFIG["PURGE_SUPPORT_OVERLAP"] if purge is None else purge
    lam = tuple(lam or V1_2_CONFIG["FIXED_LAMBDA"])
    burn_in = burn_in or V1_2_CONFIG["TARGET_BURN_IN_DAYS"]
    v1_mode = (target_stats == "whole_series_v1")
    rows, log, t_start = [], [], time.time()
    for p in participants:
        if p.get("days_df") is None:
            continue
        pr = prepare(p, feat_names, has_labels, participants)
        n = pr["n"]
        if n < 15:
            continue
        try:
            folds = get_folds(pr, g)
        except Exception:
            continue
        if not folds:
            continue
        B = min(burn_in, int(min(int(tr.max()) for tr, _ in folds)) + 1)   # burn-in inside every training window
        key = (pr["pid"], tuple(feat_names), target, target_stats, alignment, whole_series_features,
               g, purge, lam, B, target_mode)
        cached = _RUN_CACHE.get(key)
        if cached is not None and set(arms) <= set(cached["preds"]):
            res = cached
        else:
            y_fixed, idx = build_target(pr, target, "whole_series_v1" if v1_mode else "causal",
                                        alignment, lam, B)
            preds = {a: [] for a in arms}; y_all, te_all, n_purged = [], [], 0
            for tr_idx, te_idx in folds:
                if int(te_idx.min()) - int(tr_idx.max()) < g:
                    continue
                if purge and not v1_mode:
                    tr_idx, npg = purge_training_rows(tr_idx, te_idx, alignment, n); n_purged += npg
                Xf, X3, XVS = fold_features(pr, tr_idx, te_idx, whole_series_features or v1_mode,
                                            need_vsigned="vsigned" in arms)
                if target_mode == "per_fold":
                    assert target == "self" and alignment == "trailing_v1", "per-fold rebuild: trailing self-target only"
                    idx_f = ndi.compute_ndi_v3_primary(X3[:, 0], X3[:, 1], X3[:, 2], lambda_vec=lam)
                    y = smooth_target(idx_f, alignment)
                else:
                    y = y_fixed
                tr_v = tr_idx[np.isfinite(y[tr_idx])]; te_v = te_idx[np.isfinite(y[te_idx])]
                if len(tr_v) < 5 or len(te_v) < 1:
                    continue
                try:
                    fold_pred = {}
                    if "vsigned" in arms:
                        m = ndi.rf_factory(); m.fit(XVS[tr_v], y[tr_v]); fold_pred["vsigned"] = m.predict(XVS[te_v])
                    if "v3" in arms:
                        m = ndi.rf_factory(); m.fit(X3[tr_v], y[tr_v]); fold_pred["v3"] = m.predict(X3[te_v])
                    if "rawrf" in arms:
                        m = ndi.rf_factory(); m.fit(Xf[tr_v], y[tr_v]); fold_pred["rawrf"] = m.predict(Xf[te_v])
                    if "persz" in arms:
                        fold_pred["persz"] = _zscore_ridge(Xf[tr_v], y[tr_v], Xf[te_v])
                except Exception:
                    continue
                for a in arms:
                    preds[a].extend(np.asarray(fold_pred[a], float).tolist())
                y_all.extend(y[te_v].tolist()); te_all.extend(te_v.tolist())
            res = dict(preds=preds, y=y_all, te=te_all, idx=idx, y_fixed=y_fixed, B=B, n_purged=n_purged)
            _RUN_CACHE[key] = res
        if len(res["y"]) < 3:
            continue
        yt = np.array(res["y"]); row = dict(pid=pr["pid"], n_folds=len(yt))
        for a in arms:
            pa = np.array(res["preds"][a]); ok = np.isfinite(yt) & np.isfinite(pa)
            row[a] = spearmanr(yt[ok], pa[ok])[0] if ok.sum() >= 3 else np.nan
        rows.append(row)
        log.append(dict(pid=pr["pid"], burn_in_used=res["B"], purged_rows_summed_over_folds=res["n_purged"],
                        first_train_max=int(folds[0][0].max()), first_test_day=int(folds[0][1].min())))
    out = pd.DataFrame(rows)
    RUN_LOGS[label or target] = pd.DataFrame(log)
    print(f"  [{label or target}] {len(out)} participants, {time.time() - t_start:.0f}s")
    return out

def l2_proxy_rho(participants, feat_names, has_labels, lam, alignment=None, g=G, burn_in=None):
    """Leak L2: no model -- target index at the most recent available day vs the target,
    on exactly the test days the model-based runner evaluated."""
    alignment = alignment or V1_2_CONFIG["TARGET_ALIGNMENT"]
    burn_in = burn_in or V1_2_CONFIG["TARGET_BURN_IN_DAYS"]
    rows = []
    for p in participants:
        if p.get("days_df") is None:
            continue
        pr = prepare(p, feat_names, has_labels, participants)
        if pr["n"] < 15:
            continue
        folds = get_folds(pr, g)
        if not folds:
            continue
        B = min(burn_in, int(min(int(tr.max()) for tr, _ in folds)) + 1)
        y, idx = build_target(pr, "self", "causal", alignment, tuple(lam), B)
        days = np.array(sorted({int(s) for _, te in folds for s in te}))
        lag = 1 if alignment == "trailing_v1" else 0
        days = days[days >= lag]
        sc, tg = idx[days - lag], y[days]
        ok = np.isfinite(sc) & np.isfinite(tg)
        if ok.sum() >= 3:
            rows.append(dict(pid=pr["pid"], n_folds=int(ok.sum()), v3=spearmanr(sc[ok], tg[ok])[0]))
    return pd.DataFrame(rows)


In [ ]:
# V12-3 Fidelity check: the unified runner in v1 mode must reproduce Table 1 exactly
print("Fidelity check (v1 mode) ...")
v1_self = run_arms(pmdata_participants, PMDATA_FEATURE_NAMES, False, target="self",
                   target_stats="whole_series_v1", purge=False, label="v1-mode self")
v1_raw = run_arms(pmdata_participants, PMDATA_FEATURE_NAMES, False, target="raw",
                  target_stats="whole_series_v1", purge=False, label="v1-mode raw")
for new, ref, name in [(v1_self, pmdata_primary, "self-derived"), (v1_raw, pmdata_raw_target_primary, "raw-feature")]:
    new = new[new["n_folds"] >= V1_2_CONFIG["MIN_FOLDS"]].set_index("pid"); ref = ref.set_index("pid")
    assert sorted(new.index) == sorted(ref.index), f"{name}: participant set differs from v1"
    assert (new.loc[ref.index, "n_folds"] == ref["n_folds"]).all(), f"{name}: n_folds differ from v1"
    worst = max(float((new.loc[ref.index, a] - ref[a]).abs().max()) for a in ARMS)
    assert worst < 1e-9, f"{name}: max |diff| vs v1 = {worst:.2e} -- runner NOT faithful, STOP"
    print(f"  {name}: reproduces v1 exactly (max |diff| = {worst:.1e})")
print("FIDELITY CHECK PASSED")

In [ ]:
# V12-4 No-look-ahead test on the corrected path (perturb the future, nothing may move)
def no_lookahead_test(participants, feat_names, has_labels, n_participants=3, seed=0):
    rng = np.random.default_rng(seed)
    for p in participants[:n_participants]:
        pr = prepare(p, feat_names, has_labels, participants)
        folds = get_folds(pr, G)
        tr, te = folds[len(folds) // 2]
        s = int(te.max())
        _, X3a, VSa = fold_features(pr, tr, te, False, True)
        pert = dict(pr); Xn = pr["X_nan"].copy()
        Xn[s + 1:] = rng.normal(5, 5, size=Xn[s + 1:].shape)
        pert["X_nan"] = Xn
        d0p = pr["d0"].copy()
        for j, c in enumerate(feat_names):
            if c in d0p.columns:
                d0p.loc[s + 1:, c] = Xn[s + 1:, j]
        pert["d0"] = d0p
        _, X3b, VSb = fold_features(pert, tr, te, False, True)
        assert np.allclose(X3a, X3b, equal_nan=True) and np.allclose(VSa, VSb, equal_nan=True), \
            f"{pr['pid']}: fold features moved when future rows changed"
        B = min(V1_2_CONFIG["TARGET_BURN_IN_DAYS"], int(folds[0][0].max()) + 1)
        Ta, Ia, Fa, _ = causal_channels(pr, B)
        _CAUSAL_CACHE.pop((pr["pid"] + "_pert", tuple(feat_names), B), None)
        pert["pid"] = pr["pid"] + "_pert"
        Tb, Ib, Fb, _ = causal_channels(pert, B)
        upto = max(s, B - 1)
        for a, b, nm in [(Ta, Tb, "T"), (Ia, Ib, "I"), (Fa, Fb, "F")]:
            assert np.allclose(a[:upto + 1], b[:upto + 1], equal_nan=True), \
                f"{pr['pid']}: causal target channel {nm} moved when future rows changed"
    print(f"NO-LOOK-AHEAD TEST PASSED on {n_participants} participants")

no_lookahead_test(pmdata_participants, PMDATA_FEATURE_NAMES, False)

In [ ]:
# V12-5 Corrected Table 1 and raw-target comparison + v1-vs-corrected table
corr_self_all = run_arms(pmdata_participants, PMDATA_FEATURE_NAMES, False, target="self", label="corrected self")
corr_raw_all = run_arms(pmdata_participants, PMDATA_FEATURE_NAMES, False, target="raw", label="corrected raw")
MF = V1_2_CONFIG["MIN_FOLDS"]
corr_self = corr_self_all[corr_self_all["n_folds"] >= MF].reset_index(drop=True)
corr_raw = corr_raw_all[corr_raw_all["n_folds"] >= MF].reset_index(drop=True)
corr_self.to_csv(os.path.join(RESULTS_V1_2_DIR, "table1_corrected_self_target_per_participant.csv"), index=False)
corr_raw.to_csv(os.path.join(RESULTS_V1_2_DIR, "raw_target_corrected_per_participant.csv"), index=False)
RUN_LOGS["corrected self"].to_csv(os.path.join(RESULTS_V1_2_DIR, "fold_log_self_target.csv"), index=False)
RUN_LOGS["corrected raw"].to_csv(os.path.join(RESULTS_V1_2_DIR, "fold_log_raw_target.csv"), index=False)
print("Participants (n_folds >= 10): self", len(corr_self), "| raw", len(corr_raw),
      "| v1 had", len(pmdata_primary))
print(RUN_LOGS["corrected self"].head().to_string(index=False))

# Sensitivity row: self-derived target rebuilt per fold (memo Item 1)
if V1_2_CONFIG["TARGET_ALIGNMENT"] == "trailing_v1":
    sens = run_arms(pmdata_participants, PMDATA_FEATURE_NAMES, False, target="self",
                    target_mode="per_fold", label="sensitivity per-fold target")
    sens = sens[sens["n_folds"] >= MF]
    sens.to_csv(os.path.join(RESULTS_V1_2_DIR, "sensitivity_per_fold_target_per_participant.csv"), index=False)

In [ ]:
# V12-6 Task 1 statistics   [v1.3: printed to 3 decimals]
def _rng():
    return np.random.default_rng(SEED)

def _boot(x, stat, level):
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    if len(x) < 3 or np.allclose(x, x[0]):
        return np.nan, np.nan, "degenerate"
    for method in ("BCa", "percentile"):
        try:
            r = scipy_bootstrap((x,), stat, n_resamples=N_BOOT, method=method,
                                confidence_level=level, vectorized=False, rng=_rng())
        except TypeError:   # scipy < 1.15
            r = scipy_bootstrap((x,), stat, n_resamples=N_BOOT, method=method,
                                confidence_level=level, vectorized=False, random_state=_rng())
        lo, hi = float(r.confidence_interval.low), float(r.confidence_interval.high)
        if np.isfinite(lo) and np.isfinite(hi):
            return lo, hi, method
    return np.nan, np.nan, "failed"

def hodges_lehmann(d):
    d = np.asarray(d, float); n = len(d)
    return float(np.median([(d[i] + d[j]) / 2 for i in range(n) for j in range(i, n)]))

def bh(p):
    p = np.asarray(p, float); m = len(p); order = np.argsort(p)
    q = np.empty(m); prev = 1.0
    for rank, i in reversed(list(enumerate(order, start=1))):
        prev = min(prev, p[i] * m / rank); q[i] = prev
    return q

def paired_row(df, a, b, label, target):
    d = df[[a, b]].dropna(); diff = (d[a] - d[b]).values
    lo95, hi95, m95 = _boot(diff, np.mean, 0.95)
    lo90, hi90, _ = _boot(diff, np.mean, 0.90)
    hlo, hhi, _ = _boot(diff, hodges_lehmann, 0.95)
    try:
        pv = float(wilcoxon(d[a], d[b]).pvalue)
    except ValueError:
        pv = np.nan
    return dict(comparison=label, target=target, n=len(d), mean_delta_rho=float(diff.mean()),
                ci95_lo=lo95, ci95_hi=hi95, ci_method=m95,
                hl_median=hodges_lehmann(diff), hl_ci95_lo=hlo, hl_ci95_hi=hhi,
                wins=int((diff > 0).sum()), p=pv, ci90_lo=lo90, ci90_hi=hi90)

rows = []
for tgt_label, df_ in [("self-derived", corr_self), ("raw-feature", corr_raw)]:
    block = [paired_row(df_, a, b, lab, tgt_label) for a, b, lab in COMPARISONS]
    qs = bh([r["p"] for r in block])
    for r, q in zip(block, qs):
        r["q"] = q
    rows += block
    rows.append({**paired_row(df_, "v3", "vsigned", "V3 vs V-signed", tgt_label), "q": np.nan})
t2 = pd.DataFrame(rows)

# Equivalence and bound columns
is_raw = t2["target"] == "raw-feature"
t2["tost_equivalent_at_SESOI"] = np.where(is_raw, (t2["ci90_lo"] > -SESOI) & (t2["ci90_hi"] < SESOI), "n/a")
t2["advantage_ge_SESOI_rejected"] = np.where(is_raw, t2["ci90_hi"] < SESOI, "n/a")
self_adv = t2[~is_raw].set_index("comparison")
t2["self_derived_advantage"] = np.where(is_raw, t2["comparison"].map(self_adv["mean_delta_rho"]), np.nan)
t2["self_derived_advantage_q"] = np.where(is_raw, t2["comparison"].map(self_adv["q"]), np.nan)
t2["artifact_excluded"] = np.where(is_raw, t2["ci95_hi"] < t2["self_derived_advantage"], "n/a")
t2.to_csv(os.path.join(RESULTS_V1_2_DIR, "task1_table2_bounded_null.csv"), index=False)
print(t2.round(3).to_string(index=False))

# Absolute rho per arm per target
abs_rows = []
for tgt_label, df_ in [("self-derived", corr_self), ("raw-feature", corr_raw)]:
    for a in ARMS:
        v = df_[a].dropna().values
        lo, hi, _ = _boot(v, np.mean, 0.95)
        try:
            p0 = float(wilcoxon(v).pvalue)
        except ValueError:
            p0 = np.nan
        abs_rows.append(dict(arm=ARM_LABEL[a], target=tgt_label, n=len(v), mean_rho=float(v.mean()),
                             ci95_lo=lo, ci95_hi=hi, p_vs_zero=p0))
abs_df = pd.DataFrame(abs_rows)
abs_df.to_csv(os.path.join(RESULTS_V1_2_DIR, "task1_absolute_rho_both_targets.csv"), index=False)
print(abs_df.round(3).to_string(index=False))

raw_means = abs_df[abs_df["target"] == "raw-feature"]["mean_rho"]
floor_note = (f"FLOOR NOTE: on the raw-feature target, arm mean rho ranges "
              f"{raw_means.min():.3f}-{raw_means.max():.3f}"
              + (" -- every arm is below 0.10; the bound is on a target no arm predicts."
                 if (raw_means < 0.10).all() else "."))
sig_self = t2[(~is_raw) & t2["q"].notna()]
artifact_note = ("ARTIFACT NOTE: the 'artifact excluded' column is interpretable only for comparisons whose "
                 "self-derived advantage is significant (q < 0.05). Significant on the corrected path: "
                 + (", ".join(sig_self[sig_self["q"] < 0.05]["comparison"]) or "NONE"))
print(floor_note); print(artifact_note)
with open(os.path.join(RESULTS_V1_2_DIR, "task1_notes.txt"), "w") as f:
    f.write(floor_note + "\n" + artifact_note + "\n")

# v1 vs corrected, for every primary number
cmp_rows = []
for tgt_label, v1_df, c_df in [("self-derived", pmdata_primary, corr_self),
                               ("raw-feature", pmdata_raw_target_primary, corr_raw)]:
    m = v1_df.merge(c_df, on="pid", suffixes=("_v1", "_corr"))
    for a in ARMS:
        dd = (m[f"{a}_corr"] - m[f"{a}_v1"]).values
        lo, hi, _ = _boot(dd, np.mean, 0.95)
        cmp_rows.append(dict(target=tgt_label, quantity=f"mean rho {ARM_LABEL[a]}", n_paired=len(m),
                             v1=float(v1_df[a].mean()), corrected=float(c_df[a].mean()),
                             paired_change=float(dd.mean()), change_ci95_lo=lo, change_ci95_hi=hi))
    for a, b, lab in COMPARISONS:
        cmp_rows.append(dict(target=tgt_label, quantity=f"mean delta {lab}", n_paired=len(m),
                             v1=float((v1_df[a] - v1_df[b]).mean()),
                             corrected=float((c_df[a] - c_df[b]).mean())))
v1_vs_corr = pd.DataFrame(cmp_rows)
v1_vs_corr.to_csv(os.path.join(RESULTS_V1_2_DIR, "v1_vs_corrected.csv"), index=False)
print(v1_vs_corr.round(3).to_string(index=False))

In [ ]:
# V12-7 Task 2: leak states, cumulative waterfall, one-at-a-time   [v1.3: printed to 3 decimals]
LAM_FIXED, LAM_STAR = tuple(V1_2_CONFIG["FIXED_LAMBDA"]), tuple(V1_2_CONFIG["LAMBDA_STAR"])

def run_state(leaks, label):
    L = {k: (k in leaks) for k in ("L0", "L1", "L2", "L3", "L4")}   # L3 is inert by design
    g = 0 if L["L0"] else G
    lam = LAM_STAR if L["L4"] else LAM_FIXED
    # L0 removes BOTH protections against shared smoothing support: the embargo and the purge.
    df = run_arms(pmdata_participants, PMDATA_FEATURE_NAMES, False, target="self", arms=["v3", "rawrf"],
                  whole_series_features=L["L1"], g=g, purge=not L["L0"], lam=lam, label=label)
    df = df[df["n_folds"] >= MF]
    if L["L2"]:
        prox = l2_proxy_rho(pmdata_participants, PMDATA_FEATURE_NAMES, False, lam, g=g)
        df = df.drop(columns="v3").merge(prox[["pid", "v3"]], on="pid", how="inner")
    return df

def summarise_state(df, label, clean_mean=None):
    lo, hi, _ = _boot(df["v3"].values, np.mean, 0.95)
    dl = (df["v3"] - df["rawrf"]).values
    dlo, dhi, _ = _boot(dl, np.mean, 0.95)
    m = float(df["v3"].mean())
    return dict(state=label, n=len(df), v3_mean_rho=m, v3_ci95_lo=lo, v3_ci95_hi=hi,
                delta_v3_minus_rawrf=float(dl.mean()), delta_ci95_lo=dlo, delta_ci95_hi=dhi,
                inflation_vs_clean=(m - clean_mean) if clean_mean is not None else np.nan)

clean = run_state(set(), "clean")
chk = clean.merge(corr_self[["pid", "v3", "rawrf"]], on="pid", suffixes=("", "_t1"))
assert len(chk) == len(clean) == len(corr_self) and \
       np.allclose(chk["v3"], chk["v3_t1"], atol=1e-9) and np.allclose(chk["rawrf"], chk["rawrf_t1"], atol=1e-9), \
       "Task 2 clean endpoint does not equal corrected Table 1 -- STOP"
print("SANITY CHECK PASSED: Task 2 clean endpoint == corrected Table 1, participant by participant")
clean_mean = float(clean["v3"].mean())

# (A) cumulative: all on, then remove in the fixed order
order = V1_2_CONFIG["LEAK_REMOVAL_ORDER"]
on = set(order); cum_rows = [summarise_state(run_state(on, "all on"), "All leaks ON", clean_mean)]
for leak in order:
    on = on - {leak}
    lab = f"-{leak} (left: {','.join(sorted(on)) or 'none'})"
    r = summarise_state(run_state(on, lab), lab, clean_mean)
    r["removed"] = leak; r["inert"] = leak in V1_2_CONFIG["INERT_LEAKS"]
    cum_rows.append(r)
cum = pd.DataFrame(cum_rows)
cum["step_change"] = cum["v3_mean_rho"].diff()
assert abs(cum.iloc[-1]["v3_mean_rho"] - clean_mean) < 1e-12
cum.to_csv(os.path.join(RESULTS_V1_2_DIR, "task2_waterfall_cumulative.csv"), index=False)

# (B) one at a time
oat = pd.DataFrame([summarise_state(clean, "Clean (no leaks)", clean_mean)] +
                   [summarise_state(run_state({lk}, f"clean+{lk}"), f"Clean + {lk} only", clean_mean)
                    for lk in ["L0", "L1", "L2", "L3", "L4"]])
oat.to_csv(os.path.join(RESULTS_V1_2_DIR, "task2_one_at_a_time.csv"), index=False)

t4col = pd.DataFrame([dict(leak=lk, standalone_inflation=float(oat.loc[oat["state"] == f"Clean + {lk} only",
                                                                       "inflation_vs_clean"].iloc[0]),
                           note=V1_2_CONFIG["LEAK_DEFINITIONS"][lk]) for lk in ["L0", "L1", "L2", "L3", "L4"]])
t4col.to_csv(os.path.join(RESULTS_V1_2_DIR, "task2_table4_inflation_column.csv"), index=False)
print(cum.round(3).to_string(index=False)); print(oat.round(3).to_string(index=False)); print(t4col.round(3).to_string(index=False))

In [ ]:
# V12-8 (v1.3, unchanged in v1.4 except the last print) Figure 4 with plain-word labels (no figure number in the image; title goes in the caption)
import matplotlib.pyplot as plt
LEAK_PLAIN = {"L0": "no embargo (L0)", "L1": "whole-series normalisation (L1)", "L2": "self-proxy (L2)",
              "L3": "in-sample weights (L3)", "L4": "tuned-weight target (L4)"}
fig4_labels = ["All leaks"] + [f"− {LEAK_PLAIN[r['removed']]}" for r in cum_rows[1:]]
fig4_labels[-1] = f"Leak-free\n(− {LEAK_PLAIN[cum_rows[-1]['removed']]})"
fig, ax = plt.subplots(figsize=(10, 5))
colours = ["#C44E52"] + ["#BBBBBB" if r.get("inert") else "#4C72B0" for r in cum_rows[1:]]
x = np.arange(len(cum))
ax.bar(x, cum["v3_mean_rho"], color=colours)
ax.errorbar(x, cum["v3_mean_rho"],
            yerr=[cum["v3_mean_rho"] - cum["v3_ci95_lo"], cum["v3_ci95_hi"] - cum["v3_mean_rho"]],
            fmt="none", ecolor="black", capsize=4)
ax.axhline(clean_mean, ls="--", lw=1, color="crimson", label=f"Leak-free estimate, mean ρ = {clean_mean:.3f}")
ax.set_xticks(x); ax.set_xticklabels(fig4_labels, rotation=30, ha="right", fontsize=8)
ax.set_ylabel("V3 mean within-participant ρ (PMData, ≥10 test folds)")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), frameon=False, fontsize=8)
ax.set_ylim(0, max(0.8, float(cum["v3_ci95_hi"].max()) + 0.05))
fig.tight_layout()
for ext in ("png", "pdf"):
    fig.savefig(os.path.join(RESULTS_V1_2_DIR, f"figure4_leak_waterfall.{ext}"), dpi=300)
plt.show()
pd.DataFrame({"bar": fig4_labels, "state": cum["state"], "v3_mean_rho": cum["v3_mean_rho"].round(3)}).to_csv(
    os.path.join(RESULTS_V1_2_DIR, "figure4_bar_labels.csv"), index=False)
print("Grey bar = leak with no pathway into this arm (L3).")
print("All v1.4 outputs ->", RESULTS_V1_2_DIR)

## v1.4 additions: bookkeeping and data facts, independent check, supplementary without p12, comparison table, hand-in check

In [ ]:
# X-1 (v1.4) Forward-target bookkeeping, day arithmetic, v1.4 data facts (rows removed, p12), wellness table files
_need = ["V1_2_CONFIG", "RESULTS_V1_4_DIR", "RESULTS_V1_3_ARCHIVE", "corr_self", "corr_raw", "prepare", "build_target",
         "target_support", "purge_training_rows", "pmdata_participants", "DATE_CHECK", "wellness_daily", "wellness_log",
         "wellness_zeros"]
_miss = [g for g in _need if g not in globals()]
assert not _miss, f"Run V12-0 ... V12-7 first. Missing: {_miss}"
assert V1_2_CONFIG["TARGET_ALIGNMENT"] == "forward", "this cell is written for the forward target"
_k, _g, _lam = V1_2_CONFIG["SMOOTH_K"], V1_2_CONFIG["EMBARGO_G"], tuple(V1_2_CONFIG["FIXED_LAMBDA"])

# --- (a) test days removed by the full-window rule (min_periods = k), unchanged from v1.3
_rows = []
for _p in pmdata_participants:
    if _p.get("days_df") is None:
        continue
    _pr = prepare(_p, PMDATA_FEATURE_NAMES, False, pmdata_participants)
    _n, _t0 = _pr["n"], _pr["t0"]
    if _n < 15 or _t0 + _g >= _n:
        continue
    _B = min(V1_2_CONFIG["TARGET_BURN_IN_DAYS"], _t0)
    _y, _idx = build_target(_pr, "self", "causal", "forward", _lam, _B)
    _test = np.arange(_t0 + _g, _n)
    _y_mp1 = pd.Series(_idx)[::-1].rolling(_k, min_periods=1).mean()[::-1].shift(-1).values   # the v1.2-era rule
    _nan, _nan1 = int(np.isnan(_y[_test]).sum()), int(np.isnan(_y_mp1[_test]).sum())
    _rows.append(dict(pid=_pr["pid"], n_rows=_n, min_training_rows_t0=_t0, first_test_row=_t0 + _g,
                      test_rows_total=len(_test), test_rows_scored=len(_test) - _nan,
                      dropped_no_full_7_row_window=_nan, dropped_anyway_under_min_periods_1=_nan1,
                      dropped_by_min_periods_change=_nan - _nan1))
dropped = pd.DataFrame(_rows).merge(corr_self[["pid", "n_folds"]], on="pid", how="left")
_chk = dropped.dropna(subset=["n_folds"])
assert (_chk["n_folds"].astype(int) == _chk["test_rows_scored"]).all(), "scored test rows do not match n_folds in Table 1"
dropped.to_csv(os.path.join(RESULTS_V1_4_DIR, "forward_target_dropped_test_days.csv"), index=False)
print(dropped.to_string(index=False))

# --- (b) day arithmetic for the first test fold, printed from the code itself (unchanged from v1.3)
_p0 = pmdata_participants[0]
_n0 = len(_p0["days_df"]); _t0 = min(10, max(5, _n0 // 3))
_te0 = _t0 + _g; _tr0 = np.arange(0, _te0 - _g)
lines = [f"Participant {_p0['pid']}: n = {_n0} rows (= calendar days), T0 = {_t0}, g = {_g}, k = {_k}",
         f"First test fold: training rows {_tr0.min()}..{_tr0.max()} (0-based), test row {_te0}",
         "Row s holds the features of calendar day s (T, I on day s; F from days <= s)."]
for al in ("forward", "trailing_v1"):
    sup_te = sorted(target_support(_te0, al, _n0)); sup_last = sorted(target_support(int(_tr0.max()), al, _n0))
    clash = sorted(set(sup_te) & set(_tr0.tolist()))
    kept, n_drop = purge_training_rows(_tr0, np.array([_te0]), al, _n0)
    lines += [f"[{al}]" + ("  <- primary" if al == "forward" else "  <- v1.2 record"),
              f"  test target y[{_te0}] averages the index over rows {sup_te[0]}..{sup_te[-1]}",
              f"  last training row {int(_tr0.max())}: its target averages rows {sup_last[0]}..{sup_last[-1]}",
              f"  training rows inside the test-target window: {clash if clash else 'none'}",
              f"  last training target window touches the test row: {_te0 in sup_last}",
              f"  purge drops {n_drop} training row(s); last training row kept = {int(kept.max())}"
              f"  -> effective gap = {_te0 - int(kept.max()) - 1} rows"]
lines += [f"Forward target needs all {_k} following rows: the last scored test row of a series with n rows is n-{_k + 1} "
          f"(for {_p0['pid']}: row {_n0 - _k - 1}); the final {_k} rows have no target and are not scored.",
          "Rows are consecutive calendar days (cell D-2), so row counts and day counts agree.",
          f"Alignment used for every v1.4 number in this run: {V1_2_CONFIG['TARGET_ALIGNMENT']}"]
print("\n".join(lines))
with open(os.path.join(RESULTS_V1_4_DIR, "item2_day_arithmetic.txt"), "w") as f:
    f.write("\n".join(lines) + "\n")

# --- (c) v1.4 data facts: rows per participant vs v1.3, p12, wellness rules
DATE_CHECK.to_csv(os.path.join(RESULTS_V1_4_DIR, "date_check_v1_4.csv"), index=False)
_a13 = pd.read_csv(os.path.join(RESULTS_V1_3_ARCHIVE, "forward_target_dropped_test_days.csv"))
rows_cmp = (DATE_CHECK[["pid", "rows"]].rename(columns={"rows": "rows_v1_4"})
            .merge(_a13[["pid", "n_rows", "test_rows_scored"]].rename(columns={"n_rows": "rows_v1_3",
                                                                              "test_rows_scored": "scored_test_rows_v1_3"}),
                   on="pid", how="outer")
            .merge(dropped[["pid", "test_rows_scored"]].rename(columns={"test_rows_scored": "scored_test_rows_v1_4"}),
                   on="pid", how="left"))
rows_cmp["rows_removed_vs_v1_3"] = rows_cmp["rows_v1_3"] - rows_cmp["rows_v1_4"]
for _c in ["rows_v1_4", "rows_v1_3", "scored_test_rows_v1_3", "scored_test_rows_v1_4", "rows_removed_vs_v1_3"]:
    rows_cmp[_c] = rows_cmp[_c].astype("Int64")      # whole numbers; <NA> where a version has no such participant
rows_cmp["in_table1_v1_4"] = rows_cmp["pid"].isin(corr_self["pid"])
rows_cmp.to_csv(os.path.join(RESULTS_V1_4_DIR, "rows_v1_3_vs_v1_4.csv"), index=False)
print(rows_cmp.to_string(index=False))
_rm = rows_cmp.dropna(subset=["rows_removed_vs_v1_3"])
_rm = {r.pid: int(r.rows_removed_vs_v1_3) for r in _rm.itertuples() if r.rows_removed_vs_v1_3 != 0}
_p12 = rows_cmp[rows_cmp["pid"] == "p12"]
P12_LOADED = bool(len(_p12)) and pd.notna(_p12["rows_v1_4"].iloc[0])
P12_ENTERS = "p12" in set(corr_self["pid"])
if P12_LOADED:
    _r12 = int(_p12["rows_v1_4"].iloc[0])
    _s12 = _p12["scored_test_rows_v1_4"].iloc[0]
    _s12 = "n/a (fewer than 15 rows)" if pd.isna(_s12) else int(_s12)
    p12_line = (f"3. p12: loaded with the max heart rate fix; {_r12} rows; scored test folds (forward target) {_s12}; "
                f"enters the analysis under the existing rules (>= 15 rows, >= 10 scored test folds): "
                f"{'YES' if P12_ENTERS else 'NO'}.")
else:
    p12_line = "3. p12: NOT loaded in this run -- see the CELL 5 log."
_wl = wellness_log
facts = ["V1.4 DATA FACTS",
         f"1. Minimum training window: t0 = min(10, max(5, n // 3)). n ranges {int(dropped['n_rows'].min())}-"
         f"{int(dropped['n_rows'].max())} rows over {len(dropped)} participants; t0 values found: "
         f"{sorted(set(int(v) for v in dropped['min_training_rows_t0']))}.",
         f"2. Rows are calendar days: one row per local date from the Fitbit daily files, dates unique and consecutive "
         f"for all {len(DATE_CHECK)} participants (D-2). Rows removed relative to v1.3 (duplicated days): "
         f"{_rm if _rm else 'none'} (total {sum(_rm.values())}).",
         p12_line,
         f"4. Wellness table (W1-W4, cell W-1; not used in Tasks 1-2): {int(_wl['reports'].sum())} reports -> "
         f"{len(wellness_daily)} participant-days. W1 moved {int(_wl['reports_moved_by_W1'].sum())} reports to their "
         f"local date; W3 dropped {int(_wl['reports_dropped_W3'].sum())} blank reports; W2 set "
         f"{int(wellness_zeros['zeros_left_after_W3'].sum())} remaining zeros to missing; W4 chose among 2+ reports on "
         f"{int(_wl['days_with_2plus_reports_W4'].sum())} days (mood values differed on "
         f"{int(_wl['W4_days_values_differ_mood'].sum())})."]
print("\n".join(facts))
with open(os.path.join(RESULTS_V1_4_DIR, "data_facts_v1_4.txt"), "w") as f:
    f.write("\n".join(facts) + "\n")

# --- (d) wellness table and its logs (Task 3 input; the zero counts are deliverable 2)
wellness_daily.to_csv(os.path.join(RESULTS_V1_4_DIR, "wellness_daily_W1_W4.csv"), index=False)
wellness_log.to_csv(os.path.join(RESULTS_V1_4_DIR, "wellness_rules_log.csv"), index=False)
wellness_zeros.to_csv(os.path.join(RESULTS_V1_4_DIR, "wellness_zero_counts.csv"), index=False)
print(f"wellness files -> {RESULTS_V1_4_DIR}")

In [ ]:
# X-2 (v1.4) Independent recomputation of the v1.4 clean endpoint, forward alignment (no run_arms, no cache)
# Written from the memo's definitions only. Two separate checks per participant:
#   (i)  the causal FORWARD target rebuilt here from scratch equals the V12 target (to 1e-12),
#        including which rows have no target;
#   (ii) an independent fold loop (own purge rule, own feature construction, own fits), run on the
#        V12 target array, reproduces corrected Table 1 for V3 and raw-RF (to 1e-9).
# The fold loop uses nothing from the V12 runner: no run_arms, fold_features, causal_channels,
# purge_training_rows, or cache. build_target is called once per participant, only to obtain the
# V12 target array that (i) compares against. Colab: about 1 min per participant. Same code as v1.3; only the folder and labels changed.
X2_PARTICIPANTS = None      # None = all; or e.g. ["p01", "p08", "p16"] for a quick pass

def _x2_fill(X, rows):
    """NaN -> column median over `rows` (all-NaN column -> 0.0)."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        med = np.nanmedian(X[rows], axis=0)
    med = np.where(np.isfinite(med), med, 0.0)
    return np.where(np.isnan(X), med, X)

def _x2_inputs(p, participants, feat_names):
    d0 = p["days_df"].copy().reset_index(drop=True); n = len(d0)
    Xn = np.column_stack([pd.to_numeric(d0[c], errors="coerce").values if c in d0.columns
                          else np.zeros(n) for c in feat_names]).astype(float)
    pm, pmad, _ = _compute_population_volatility_norm(participants, p["pid"], feat_names, False)
    return d0, n, Xn, pm, pmad, min(10, max(5, n // 3))

def x2_forward_target(p, participants, feat_names, k, lam, burn):
    """Index day d from rows [0, max(d+1, B)) only; target y[s] = mean(index[s+1 .. s+k]), NaN unless all k rows exist."""
    d0, n, Xn, pm, pmad, t0 = _x2_inputs(p, participants, feat_names)
    B = min(burn, t0)                                   # burn-in lies inside the first training window
    Xc = Xn.copy()
    for d in range(n):                                  # causal imputation
        if np.isnan(Xn[d]).any():
            L = min(max(d + 1, B), n)
            Xc[d] = _x2_fill(Xn[:L], np.arange(L))[d]
    idx = np.full(n, np.nan)
    for d in range(n):
        L = min(max(d + 1, B), n); ones = np.ones(L, bool)
        T = ndi.compute_T_v3(Xc[:L], ones)[d]
        mi, ii = ndi.compute_quality_signal(d0.iloc[:L].reset_index(drop=True), feat_names, ones)
        I = ndi.compute_I_v3(mi, ii)[d]
        F = ndi.compute_F_v3(Xc[:L], feat_names, ones, pm, pmad)[d]
        idx[d] = lam[2] * F - lam[0] * T + lam[1] * I   # NDI = lF*F - lT*T + lI*I, lam = (lT, lI, lF)
    y = np.array([idx[s + 1:s + k + 1].mean() if s + k <= n - 1 else np.nan for s in range(n)])
    return y, B

def x2_fold_rho(p, participants, feat_names, y, k, g):
    """Test row ts; training rows 0..ts-g-1 minus the purged ones. Forward alignment: the target of
    training row t uses rows t+1..t+k, which must not reach the test row ts (nor its target window
    ts+1..ts+k), so t <= ts-k-1. With g = 6 and k = 7 this drops exactly the last training row.
    Features: training-fold statistics, truncated at the test row."""
    d0, n, Xn, pm, pmad, t0 = _x2_inputs(p, participants, feat_names)
    yt, p_v3, p_raw = [], [], []
    for ts in range(t0 + g, n):
        tr = np.arange(0, min(ts - g, ts - k))
        L = ts + 1
        tm = np.zeros(L, bool); tm[tr] = True
        X = _x2_fill(Xn[:L], tm)
        T = ndi.compute_T_v3(X, tm)
        mi, ii = ndi.compute_quality_signal(d0.iloc[:L].reset_index(drop=True), feat_names, tm)
        I = ndi.compute_I_v3(mi, ii)
        F = ndi.compute_F_v3(X, feat_names, tm, pm, pmad)
        X3 = np.column_stack([T, I, F])
        trv = tr[np.isfinite(y[tr])]
        if len(trv) < 5 or not np.isfinite(y[ts]):
            continue
        m = ndi.rf_factory(); m.fit(X3[trv], y[trv]); p_v3.append(float(m.predict(X3[[ts]])[0]))
        m = ndi.rf_factory(); m.fit(X[trv], y[trv]);  p_raw.append(float(m.predict(X[[ts]])[0]))
        yt.append(y[ts])
    return len(yt), spearmanr(yt, p_v3)[0], spearmanr(yt, p_raw)[0]

assert V1_2_CONFIG["TARGET_ALIGNMENT"] == "forward", "this cell is written for the forward target"
_k, _g, _lam = V1_2_CONFIG["SMOOTH_K"], V1_2_CONFIG["EMBARGO_G"], tuple(V1_2_CONFIG["FIXED_LAMBDA"])
_t = time.time(); x2_rows = []
for p in pmdata_participants:
    if p.get("days_df") is None or p["pid"] not in set(corr_self["pid"]):
        continue
    if X2_PARTICIPANTS is not None and p["pid"] not in X2_PARTICIPANTS:
        continue
    y_ind, B = x2_forward_target(p, pmdata_participants, PMDATA_FEATURE_NAMES, _k, _lam,
                                 V1_2_CONFIG["TARGET_BURN_IN_DAYS"])
    y_v12, _ = build_target(prepare(p, PMDATA_FEATURE_NAMES, False, pmdata_participants),
                            "self", "causal", "forward", _lam, B)
    assert (np.isnan(y_ind) == np.isnan(y_v12)).all(), f"{p['pid']}: rows without a target differ"
    nf, r3, rr = x2_fold_rho(p, pmdata_participants, PMDATA_FEATURE_NAMES, y_v12, _k, _g)
    x2_rows.append(dict(pid=p["pid"], rows_without_target=int(np.isnan(y_ind).sum()),
                        target_max_abs_diff=float(np.nanmax(np.abs(y_ind - y_v12))),
                        n_folds_indep=nf, v3_indep=r3, rawrf_indep=rr))
x2 = pd.DataFrame(x2_rows).merge(corr_self[["pid", "n_folds", "v3", "rawrf"]], on="pid")
x2 = x2.rename(columns={"n_folds": "n_folds_v14", "v3": "v3_v14", "rawrf": "rawrf_v14"})
x2["abs_diff_v3"] = (x2["v3_indep"] - x2["v3_v14"]).abs()
x2["abs_diff_rawrf"] = (x2["rawrf_indep"] - x2["rawrf_v14"]).abs()
x2.to_csv(os.path.join(RESULTS_V1_4_DIR, "independent_check_clean_endpoint.csv"), index=False)
print(x2.round(3).to_string(index=False))
_worst = float(max(x2["abs_diff_v3"].max(), x2["abs_diff_rawrf"].max()))
x2_lines = [f"{len(x2)} participants, {time.time() - _t:.0f}s, alignment = forward",
            f"(i)  forward target rebuilt from scratch vs V12 target: max |diff| = {x2['target_max_abs_diff'].max():.1e}; "
            f"rows without a target per participant: {sorted(set(x2['rows_without_target']))}",
            f"(ii) independent fold loop vs corrected Table 1 (v1.4): max |diff| = {_worst:.1e}"]
print("\n" + "\n".join(x2_lines))
assert x2["target_max_abs_diff"].max() < 1e-12, "forward target does NOT match -- STOP and report"
assert (x2["n_folds_indep"] == x2["n_folds_v14"]).all(), "fold counts differ -- STOP and report"
assert _worst < 1e-9, "independent fold loop does NOT match corrected Table 1 -- STOP and report"
x2_lines.append("INDEPENDENT CHECK PASSED: v1.4 clean endpoint (V3, raw-RF) reproduced without the V12 runner")
print(x2_lines[-1])
with open(os.path.join(RESULTS_V1_4_DIR, "independent_check_notes.txt"), "w") as f:
    f.write("\n".join(x2_lines) + "\n")

In [ ]:
# X-6 (v1.4) Supplementary row: the v1.4 primary numbers WITHOUT p12 (v1.4 memo, Section 5)
# Run whenever p12 is loaded: p12 then sits in every other participant's donor pool (the F channel's population
# volatility norm uses all other participants) even if it misses Table 1 itself. p12 is removed from the cohort AND
# from every donor pool, so the whole primary path is re-run, not just p12's row dropped.
# The runner's caches are keyed by participant, not by cohort: they are set aside here and restored afterwards,
# so no other cell sees a p12-free donor pool. About 50 minutes on Colab when p12 enters.
_need = ["run_arms", "corr_self", "corr_raw", "paired_row", "bh", "_boot", "_PREP_CACHE", "_CAUSAL_CACHE", "_RUN_CACHE",
         "RESULTS_V1_4_DIR", "COMPARISONS", "ARMS", "ARM_LABEL", "MF"]
_miss = [g for g in _need if g not in globals()]
assert not _miss, f"Run V12-0 ... V12-7 first. Missing: {_miss}"
P12_ENTERS = "p12" in set(corr_self["pid"]) or "p12" in set(corr_raw["pid"])
P12_LOADED = any(p["pid"] == "p12" for p in pmdata_participants)
supp_self = supp_raw = None
if not P12_LOADED:
    _note = "p12 is not loaded in this run, so there is no supplementary row without p12."
else:
    _cohort = [p for p in pmdata_participants if p["pid"] != "p12"]
    _saved = [dict(c) for c in (_PREP_CACHE, _CAUSAL_CACHE, _RUN_CACHE)]
    for _c in (_PREP_CACHE, _CAUSAL_CACHE, _RUN_CACHE):
        _c.clear()
    try:
        _s = run_arms(_cohort, PMDATA_FEATURE_NAMES, False, target="self", label="without p12 self")
        _r = run_arms(_cohort, PMDATA_FEATURE_NAMES, False, target="raw", label="without p12 raw")
    finally:
        for _c, _sv in zip((_PREP_CACHE, _CAUSAL_CACHE, _RUN_CACHE), _saved):
            _c.clear(); _c.update(_sv)
    supp_self = _s[_s["n_folds"] >= MF].reset_index(drop=True)
    supp_raw = _r[_r["n_folds"] >= MF].reset_index(drop=True)
    assert set(supp_self["pid"]) == set(corr_self["pid"]) - {"p12"}, "participant set without p12 is not Table 1 minus p12"
    supp_self.to_csv(os.path.join(RESULTS_V1_4_DIR, "supplementary_without_p12_self_target_per_participant.csv"), index=False)
    supp_raw.to_csv(os.path.join(RESULTS_V1_4_DIR, "supplementary_without_p12_raw_target_per_participant.csv"), index=False)
    _rows = []
    for _tl, _df in [("self-derived", supp_self), ("raw-feature", supp_raw)]:
        _blk = [paired_row(_df, a, b, lab, _tl) for a, b, lab in COMPARISONS]
        for _row, _q in zip(_blk, bh([r["p"] for r in _blk])):
            _row["q"] = _q
        _rows += _blk + [{**paired_row(_df, "v3", "vsigned", "V3 vs V-signed", _tl), "q": np.nan}]
    supp_t2 = pd.DataFrame(_rows)
    supp_t2.to_csv(os.path.join(RESULTS_V1_4_DIR, "supplementary_without_p12_table2.csv"), index=False)
    _ab = []
    for _tl, _df in [("self-derived", supp_self), ("raw-feature", supp_raw)]:
        for a in ARMS:
            v = _df[a].dropna().values; lo, hi, _ = _boot(v, np.mean, 0.95)
            try:
                p0 = float(wilcoxon(v).pvalue)
            except ValueError:
                p0 = np.nan
            _ab.append(dict(arm=ARM_LABEL[a], target=_tl, n=len(v), mean_rho=float(v.mean()), ci95_lo=lo, ci95_hi=hi, p_vs_zero=p0))
    supp_abs = pd.DataFrame(_ab)
    supp_abs.to_csv(os.path.join(RESULTS_V1_4_DIR, "supplementary_without_p12_absolute_rho.csv"), index=False)
    print(supp_abs.round(3).to_string(index=False)); print(supp_t2.round(3).to_string(index=False))
    _note = (("p12 entered v1.4 Table 1. " if P12_ENTERS else
              "p12 loaded but is not in v1.4 Table 1 (existing rules); it is still in the other participants' donor pools. ")
             + f"Supplementary: the primary path re-run without p12 (cohort and donor pools), "
             f"n = {len(supp_self)} (self-derived), {len(supp_raw)} (raw-feature). Files: supplementary_without_p12_*.")
print(_note)
with open(os.path.join(RESULTS_V1_4_DIR, "supplementary_without_p12_notes.txt"), "w") as f:
    f.write(_note + "\n")

In [ ]:
# X-5 (v1.4) Comparison table extended with a v1.4 column (v1 | v1.2 | v1.3 | v1.4), Task 2 by version,
#              and the exploratory target-swap shift on v1.4
# v1, v1.2, v1.3 columns and the two v1.3 effects are taken unchanged from the archived v1.3 table (results_v1.3,
# read-only). New: the v1.4 column, and the de-duplication effect (paired over the participants in both versions).
# If p12 entered: also v1.4 without p12 (X-6); the de-duplication effect is then v1.4-without-p12 minus v1.3 (same
# participants, same donor pools apart from the duplicated rows), and the participant change is v1.4 minus
# v1.4-without-p12 (difference of means; the participant sets differ, so it is not paired).
_need = ["corr_self", "corr_raw", "_boot", "bh", "oat", "cum", "RESULTS_V1_4_DIR", "RESULTS_V1_3_ARCHIVE",
         "RESULTS_V1_2_ARCHIVE", "supp_self", "supp_raw"]
_miss = [g for g in _need if g not in globals()]
assert not _miss, f"Run V12-0 ... V12-7 and X-6 first. Missing: {_miss}"
_ND = V1_2_CONFIG.get("REPORTING_DECIMALS", 3)
def _fmt(m, lo, hi):
    return f"{m:.{_ND}f} [{lo:.{_ND}f}, {hi:.{_ND}f}]"
def _mci(x):
    x = np.asarray(x, float); x = x[np.isfinite(x)]; lo, hi, _ = _boot(x, np.mean, 0.95)
    return float(np.mean(x)), lo, hi

_tw = pd.read_csv(os.path.join(RESULTS_V1_3_ARCHIVE, "three_way_v1_v1_2_v1_3.csv"))
_twf = pd.read_csv(os.path.join(RESULTS_V1_3_ARCHIVE, "three_way_v1_v1_2_v1_3_formatted.csv"))
_v13 = {"self-derived": pd.read_csv(os.path.join(RESULTS_V1_3_ARCHIVE, "table1_corrected_self_target_per_participant.csv")),
        "raw-feature": pd.read_csv(os.path.join(RESULTS_V1_3_ARCHIVE, "raw_target_corrected_per_participant.csv"))}
_v14 = {"self-derived": corr_self, "raw-feature": corr_raw}
_v14np = {"self-derived": supp_self, "raw-feature": supp_raw} if supp_self is not None else None
_GET = {f"mean rho {ARM_LABEL[a]}": (lambda df, a=a: df[a]) for a in ARMS}
_GET.update({f"mean delta {lab}": (lambda df, a=a, b=b: df[a] - df[b]) for a, b, lab in COMPARISONS})

def _paired(new, old, getter):
    m = new[["pid"] + ARMS].merge(old[["pid"] + ARMS], on="pid", suffixes=("_n", "_o"))
    n_ = getter(m.rename(columns={f"{a}_n": a for a in ARMS})).values
    o_ = getter(m.rename(columns={f"{a}_o": a for a in ARMS})).values
    return n_ - o_, len(m)

rows, frows = [], []
for i, r in _tw.iterrows():
    tgt, qty = r["target"], r["quantity"]
    get = _GET[qty]
    row = r.to_dict(); frow = _twf.iloc[i].to_dict()
    assert frow["Target"] == tgt and frow["Quantity"] == qty, "archived tables are not aligned"
    m, lo, hi = _mci(get(_v14[tgt]).values)
    row.update(v1_4=m, v1_4_ci95_lo=lo, v1_4_ci95_hi=hi, n_v1_4=len(_v14[tgt]))
    frow["v1.4 (forward, corrected, one row per date)"] = _fmt(m, lo, hi) + f" (n={len(_v14[tgt])})"
    if _v14np is not None:
        mn, lon, hin = _mci(get(_v14np[tgt]).values)
        row.update(v1_4_without_p12=mn, v1_4_without_p12_ci95_lo=lon, v1_4_without_p12_ci95_hi=hin)
        frow["v1.4 without p12 (supplementary)"] = _fmt(mn, lon, hin) + f" (n={len(_v14np[tgt])})"
        d, npair = _paired(_v14np[tgt], _v13[tgt], get)
        e_lab = "De-duplication effect (v1.4 without p12 - v1.3)"
    else:
        d, npair = _paired(_v14[tgt], _v13[tgt], get)
        e_lab = "De-duplication effect (v1.4 - v1.3)"
    em, elo, ehi = _mci(d)
    row.update(dedup_effect=em, dedup_effect_ci95_lo=elo, dedup_effect_ci95_hi=ehi, dedup_effect_n_paired=npair)
    frow[e_lab] = _fmt(em, elo, ehi) + f" (n={npair})"
    if _v14np is not None:
        row["participant_change_v1_4_minus_without_p12"] = m - mn
        frow["Participant change (v1.4 - v1.4 without p12), means"] = f"{m - mn:.{_ND}f}"
    rows.append(row); frows.append(frow)
comparison = pd.DataFrame(rows); comparison_fmt = pd.DataFrame(frows)
_first = ["Target", "Quantity", "n", "v1 (trailing, whole-series)", "v1.2 (trailing, corrected)",
          "v1.3 (forward, corrected)", "v1.4 (forward, corrected, one row per date)", "v1.4 without p12 (supplementary)"]
comparison_fmt = comparison_fmt[[c for c in _first if c in comparison_fmt.columns] +
                                [c for c in comparison_fmt.columns if c not in _first]]
comparison_fmt = comparison_fmt.rename(columns={"n": "n (v1-v1.3)"})
comparison.to_csv(os.path.join(RESULTS_V1_4_DIR, "comparison_v1_v1_2_v1_3_v1_4.csv"), index=False)
comparison_fmt.to_csv(os.path.join(RESULTS_V1_4_DIR, "comparison_v1_v1_2_v1_3_v1_4_formatted.csv"), index=False)
for _tgt in _v14:
    _nf = {"v1_3": int(_v13[_tgt]["n_folds"].sum()), "v1_4": int(_v14[_tgt]["n_folds"].sum())}
    print(f"{_tgt}: scored test rows summed over participants -> v1.3 {_nf['v1_3']}, v1.4 {_nf['v1_4']}")
print("\nCOMPARISON BY VERSION, mean [BCa 95% CI]; v1-v1.3 columns copied from the archived v1.3 table")
print(comparison_fmt.to_string(index=False))

# Task 2 by version: v1.2 (trailing), v1.3 (forward), v1.4 (forward, one row per date)
_o12 = pd.read_csv(os.path.join(RESULTS_V1_2_ARCHIVE, "task2_one_at_a_time.csv")).set_index("state")
_c12 = pd.read_csv(os.path.join(RESULTS_V1_2_ARCHIVE, "task2_waterfall_cumulative.csv")).set_index("state")
_o13 = pd.read_csv(os.path.join(RESULTS_V1_3_ARCHIVE, "task2_one_at_a_time.csv")).set_index("state")
_c13 = pd.read_csv(os.path.join(RESULTS_V1_3_ARCHIVE, "task2_waterfall_cumulative.csv")).set_index("state")
_t2 = []
for _name, _a, _b, _new in [("one at a time", _o12, _o13, oat.set_index("state")), ("cumulative", _c12, _c13, cum.set_index("state"))]:
    for _s in _new.index:
        _row = dict(table=_name, state=_s)
        for _lab, _df in [("v1_2", _a), ("v1_3", _b), ("v1_4", _new)]:
            _row[f"v3_mean_rho_{_lab}"] = float(_df.loc[_s, "v3_mean_rho"]) if _s in _df.index else np.nan
        for _lab, _df in [("v1_2", _a), ("v1_3", _b), ("v1_4", _new)]:
            _row[f"inflation_vs_clean_{_lab}"] = float(_df.loc[_s, "inflation_vs_clean"]) if _s in _df.index else np.nan
        _t2.append(_row)
task2_cmp = pd.DataFrame(_t2)
task2_cmp.to_csv(os.path.join(RESULTS_V1_4_DIR, "task2_v1_2_v1_3_v1_4.csv"), index=False)
print("\nTASK 2 BY VERSION"); print(task2_cmp.round(_ND).to_string(index=False))

# --- EXPLORATORY: target-swap shift on v1.4 (unchanged definition)
_ms = corr_self[["pid"] + ARMS].merge(corr_raw[["pid"] + ARMS], on="pid", suffixes=("_self", "_raw"))
swap_pp = _ms[["pid"]].copy(); _sw = []
for a, b, lab in COMPARISONS:
    d = ((_ms[f"{a}_self"] - _ms[f"{b}_self"]) - (_ms[f"{a}_raw"] - _ms[f"{b}_raw"])).values
    swap_pp[lab] = d
    d = d[np.isfinite(d)]
    lo, hi, meth = _boot(d, np.mean, 0.95)
    try:
        pv = float(wilcoxon(d).pvalue)
    except ValueError:
        pv = np.nan
    _sw.append(dict(comparison=lab, n=len(d), mean_shift=float(d.mean()), ci95_lo=lo, ci95_hi=hi, ci_method=meth,
                    n_positive=int((d > 0).sum()), p=pv))
swap = pd.DataFrame(_sw); swap["q"] = bh(swap["p"].values)
swap.to_csv(os.path.join(RESULTS_V1_4_DIR, "exploratory_target_swap_shift.csv"), index=False)
swap_pp.to_csv(os.path.join(RESULTS_V1_4_DIR, "exploratory_target_swap_shift_per_participant.csv"), index=False)
swap_note = ("EXPLORATORY (not a primary or pre-specified endpoint; added by the v1.3 memo before the v1.3 run): "
             "target-swap shift = (neutrosophic minus baseline on the self-derived target) minus (the same on the "
             "raw-feature target), per participant. Mean, BCa 95% CI (10,000 resamples, fixed seed), one-sample "
             "Wilcoxon P, BH q across the four comparisons. A positive shift means the advantage is larger on the "
             "target the index helped build.")
with open(os.path.join(RESULTS_V1_4_DIR, "exploratory_notes.txt"), "w") as f:
    f.write(swap_note + "\n")
print("\n" + swap_note); print(swap.round(_ND).to_string(index=False))

In [ ]:
# X-4 (v1.4) Hand-in check: files, provenance, archives untouched, checks, data facts, report tables to 3 decimals
# Changed from v1.3: folder results_v1.4; results_v1.2, results_v1.3 AND the frozen v1 archive must be untouched;
# date / p12 / wellness / supplementary checks added. The v1 regeneration check of v1.3 (this session's v1 re-run
# equals the frozen v1 archive) cannot hold on purpose -- v1.4 changes the data the v1 cells read -- so it is
# reported as information, not as a check. The fidelity check (V12-3) still requires the runner in v1 mode to
# reproduce this session's v1 cells exactly.
import os, json, datetime
import numpy as np, pandas as pd
_BASE = os.path.dirname(RESULTS_DIR)
_V14 = os.path.join(_BASE, "results_v1.4"); _V13A = os.path.join(_BASE, "results_v1.3"); _V12A = os.path.join(_BASE, "results_v1.2")
_FROZEN = os.path.join(_BASE, "results_final_v1_frozen_backup"); _REGEN = os.path.join(_BASE, "results_regen_scratch_v1_4")
assert os.path.isdir(_V14), f"{_V14} not found"
report, fails = [], []
def say(s=""):
    print(s); report.append(s)
def check(name, ok):
    say(f"   {'PASS' if ok else 'FAIL'}  {name}")
    if not ok:
        fails.append(name)

# --- 1. files (written by the cells above; the formatted tables are written further down in this cell)
_expected = ["config_v1_4.json", "provenance.json",
             "table1_corrected_self_target_per_participant.csv", "raw_target_corrected_per_participant.csv",
             "v1_vs_corrected.csv", "fold_log_self_target.csv", "fold_log_raw_target.csv",
             "task1_table2_bounded_null.csv", "task1_absolute_rho_both_targets.csv", "task1_notes.txt",
             "task2_waterfall_cumulative.csv", "task2_one_at_a_time.csv", "task2_table4_inflation_column.csv",
             "figure4_leak_waterfall.png", "figure4_leak_waterfall.pdf", "figure4_bar_labels.csv",
             "forward_target_dropped_test_days.csv", "item2_day_arithmetic.txt",
             "data_facts_v1_4.txt", "date_check_v1_4.csv", "rows_v1_3_vs_v1_4.csv",
             "wellness_daily_W1_W4.csv", "wellness_rules_log.csv", "wellness_zero_counts.csv",
             "independent_check_clean_endpoint.csv", "independent_check_notes.txt",
             "supplementary_without_p12_notes.txt",
             "comparison_v1_v1_2_v1_3_v1_4.csv", "comparison_v1_v1_2_v1_3_v1_4_formatted.csv", "task2_v1_2_v1_3_v1_4.csv",
             "exploratory_target_swap_shift.csv", "exploratory_target_swap_shift_per_participant.csv",
             "exploratory_notes.txt"]
_t1 = pd.read_csv(os.path.join(_V14, "table1_corrected_self_target_per_participant.csv"))
_p12_in = "p12" in set(_t1["pid"])
_rc0 = pd.read_csv(os.path.join(_V14, "rows_v1_3_vs_v1_4.csv")) if os.path.isfile(os.path.join(_V14, "rows_v1_3_vs_v1_4.csv")) else pd.DataFrame(columns=["pid", "rows_v1_4"])
_p12_loaded = bool(((_rc0["pid"] == "p12") & _rc0["rows_v1_4"].notna()).any())
if _p12_loaded:
    _expected += ["supplementary_without_p12_self_target_per_participant.csv",
                  "supplementary_without_p12_raw_target_per_participant.csv",
                  "supplementary_without_p12_table2.csv", "supplementary_without_p12_absolute_rho.csv"]
say("1. FILES in results_v1.4")
_missing = [f for f in _expected if not os.path.isfile(os.path.join(_V14, f))]
check(f"{len(_expected) - len(_missing)} of {len(_expected)} present" + (f"; MISSING: {_missing}" if _missing else ""), not _missing)
check("no per-fold sensitivity file (not applicable under forward alignment)",
      not os.path.isfile(os.path.join(_V14, "sensitivity_per_fold_target_per_participant.csv")))

# --- 2. provenance and configuration
say("2. PROVENANCE")
_prov = json.load(open(os.path.join(_V14, "provenance.json")))
_cfg = json.load(open(os.path.join(_V14, "config_v1_4.json")))
say(f"   run_started {_prov.get('run_started')} | tag {_prov.get('tag')} -> {_prov.get('tag_commit')} | "
    f"tag commit time {_prov.get('tag_commit_time')} | HEAD {_prov.get('head_commit')}")
check("tag commit recorded", bool(_prov.get("tag_commit")))
check("run made from the tagged commit (HEAD = tag commit)", _prov.get("head_commit") == _prov.get("tag_commit"))
check("tag commit time precedes run_started", bool(_prov.get("tag_time_precedes_run_start")))
check("config used equals the config committed under the tag", bool(_prov.get("config_matches_tagged_config")))
check("VERSION = v1.4, TARGET_ALIGNMENT = forward, W1-W4 and p12 decision in the config",
      _cfg.get("VERSION") == "v1.4" and _cfg.get("TARGET_ALIGNMENT") == "forward"
      and {"W1", "W2", "W3", "W4"} <= set(_cfg.get("WELLNESS_RULES", {})) and bool(_cfg.get("P12_DECISION")))
check(f"F_SCALE = {_prov.get('f_scale')} (frozen 1.0)", _prov.get("f_scale") == 1.0)

# --- 3. earlier records must be untouched by this run
say("3. ARCHIVES untouched since this run started")
_start = datetime.datetime.fromisoformat(_prov["run_started"]).timestamp()
for _d in (_V12A, _V13A, _FROZEN):
    if os.path.isdir(_d):
        _late = [f for f in os.listdir(_d) if os.path.getmtime(os.path.join(_d, f)) >= _start]
        check(f"{os.path.basename(_d)}: no file modified ({len(os.listdir(_d))} files)", not _late)
        if _late:
            say(f"      modified: {_late}")
    else:
        check(f"{os.path.basename(_d)} folder found", False)

# --- 4. all in-notebook checks ran and passed (each one stops the run on failure)
say("4. CHECKS")
_ic = pd.read_csv(os.path.join(_V14, "independent_check_clean_endpoint.csv"))
check("fidelity check, no-look-ahead test, Task 2 clean-endpoint check, D-2 date check, W-1 wellness check "
      "(asserts in V12-3, V12-4, V12-7, D-2, W-1: run reached this cell)", True)
check(f"independent check: {len(_ic)} of {len(_t1)} participants, max |diff| = "
      f"{max(_ic['abs_diff_v3'].max(), _ic['abs_diff_rawrf'].max()):.1e}",
      len(_ic) == len(_t1) and max(_ic["abs_diff_v3"].max(), _ic["abs_diff_rawrf"].max()) < 1e-9)
_dr = pd.read_csv(os.path.join(_V14, "forward_target_dropped_test_days.csv"))
check(f"forward target: {sorted(set(_dr['dropped_no_full_7_row_window']))} test rows dropped per participant, "
      f"scored rows equal n_folds", bool((_dr.dropna(subset=["n_folds"])["test_rows_scored"]
                                           == _dr.dropna(subset=["n_folds"])["n_folds"]).all()))

# --- 5. v1.4 data facts
say("5. DATA (v1.4 memo, Sections 2-4)")
_dc = pd.read_csv(os.path.join(_V14, "date_check_v1_4.csv"))
check(f"one row per calendar date: {len(_dc)} participants, 0 duplicated, 0 non-consecutive, 0 unparseable, "
      f"no wellness column in days_df",
      int(_dc[["duplicated_dates", "non_consecutive_steps", "unparseable_dates", "wellness_columns_in_days_df"]].values.sum()) == 0)
_rc = pd.read_csv(os.path.join(_V14, "rows_v1_3_vs_v1_4.csv"))
_rm = _rc.dropna(subset=["rows_removed_vs_v1_3"])
say(f"   rows removed relative to v1.3: {int(_rm['rows_removed_vs_v1_3'].sum())} in "
    f"{int((_rm['rows_removed_vs_v1_3'] > 0).sum())} participants (the duplicated days)")
check("no participant gained rows relative to v1.3", bool((_rm["rows_removed_vs_v1_3"] >= 0).all()))
_r12 = _rc[_rc["pid"] == "p12"]
say(f"   p12: loaded {bool(len(_r12)) and pd.notna(_r12['rows_v1_4'].iloc[0])}"
    + (f", rows {int(_r12['rows_v1_4'].iloc[0])}, scored test folds {_r12['scored_test_rows_v1_4'].iloc[0]}, "
       f"in Table 1: {_p12_in}" if len(_r12) and pd.notna(_r12['rows_v1_4'].iloc[0]) else ""))
check("supplementary row without p12 present if p12 is loaded", (not _p12_loaded) or
      os.path.isfile(os.path.join(_V14, "supplementary_without_p12_table2.csv")))
_wd = pd.read_csv(os.path.join(_V14, "wellness_daily_W1_W4.csv"))
check(f"wellness table: one row per participant and local date ({len(_wd)} rows)", not _wd.duplicated(["pid", "date"]).any())

# --- 6. information only: the v1 cells of this session read the v1.4 data, so they differ from the frozen v1 record
say("6. INFORMATION (not a check): frozen v1 record vs this session's v1 cells on the v1.4 data")
_fz = os.path.join(_FROZEN, "primary_pmdata_rho.csv"); _rg = os.path.join(_REGEN, "primary_pmdata_rho.csv")
if os.path.isfile(_fz) and os.path.isfile(_rg):
    _m = pd.read_csv(_fz).merge(pd.read_csv(_rg), on="pid", suffixes=("_frozen", "_v14data"))
    _ARMS = ["vsigned", "v3", "rawrf", "persz"]
    _dpp = max(float((_m[f"{a}_frozen"] - _m[f"{a}_v14data"]).abs().max()) for a in _ARMS)
    say(f"   {len(_m)} participants in both; largest per-participant |difference| = {_dpp:.3f} "
        "(expected to be non-zero: duplicated rows removed" + (", p12 added to the donor pools)" if _p12_loaded else ")"))
else:
    say("   frozen or v1.4-session primary_pmdata_rho.csv not found")

# --- 7. report tables, three decimals at most (unchanged from v1.3)
_ND = 3
def _f(x):
    return "n/a" if pd.isna(x) else f"{x:.{_ND}f}"
def _flag(v, yes, no):
    s = str(v).strip().lower()
    return yes if s == "true" else no if s == "false" else "n/a"
def _ci(m, l, h):
    return f"{_f(m)} [{_f(l)}, {_f(h)}]"
t2 = pd.read_csv(os.path.join(_V14, "task1_table2_bounded_null.csv"), keep_default_na=False, na_values=[""])
for c in ["mean_delta_rho", "ci95_lo", "ci95_hi", "hl_median", "hl_ci95_lo", "hl_ci95_hi", "p", "q", "ci90_lo", "ci90_hi"]:
    t2[c] = pd.to_numeric(t2[c], errors="coerce")
table2_fmt = pd.DataFrame({
    "Comparison": t2["comparison"], "Target": t2["target"],
    "Mean delta rho [95% CI]": [_ci(m, l, h) for m, l, h in zip(t2["mean_delta_rho"], t2["ci95_lo"], t2["ci95_hi"])],
    "HL median [95% CI]": [_ci(m, l, h) for m, l, h in zip(t2["hl_median"], t2["hl_ci95_lo"], t2["hl_ci95_hi"])],
    "Wins/n": [f"{w}/{n}" for w, n in zip(t2["wins"], t2["n"])],
    "P": [_f(v) for v in t2["p"]], "q (BH)": [_f(v) for v in t2["q"]],
    "TOST (+/-0.10), 90% CI": [("n/a" if _flag(e, "y", "n") == "n/a" else
                               f"{_flag(e, 'equivalent', 'inconclusive')} [{_f(l)}, {_f(h)}]")
                              for e, l, h in zip(t2["tost_equivalent_at_SESOI"], t2["ci90_lo"], t2["ci90_hi"])],
    "Advantage >= 0.10 rejected": [_flag(v, "yes", "no") for v in t2["advantage_ge_SESOI_rejected"]],
    "Artifact excluded": [_flag(v, "yes", "no") for v in t2["artifact_excluded"]]})
ab = pd.read_csv(os.path.join(_V14, "task1_absolute_rho_both_targets.csv"))
abs_fmt = pd.DataFrame({"Arm": ab["arm"], "Target": ab["target"], "n": ab["n"],
                        "Mean rho [95% CI]": [_ci(m, l, h) for m, l, h in zip(ab["mean_rho"], ab["ci95_lo"], ab["ci95_hi"])],
                        "P vs 0": [_f(v) for v in ab["p_vs_zero"]]})
_PLAIN = {"L0": "no embargo (L0)", "L1": "whole-series normalisation (L1)", "L2": "self-proxy (L2)",
          "L3": "in-sample weights (L3)", "L4": "tuned-weight target (L4)"}
cu = pd.read_csv(os.path.join(_V14, "task2_waterfall_cumulative.csv"))
_steps = ["All leaks"] + [f"- {_PLAIN[r]}" for r in cu["removed"].iloc[1:]]
_steps[-1] = "Leak-free (" + _steps[-1] + ")"
cum_fmt = pd.DataFrame({"Step": _steps,
                        "V3 mean rho [95% CI]": [_ci(m, l, h) for m, l, h in zip(cu["v3_mean_rho"], cu["v3_ci95_lo"], cu["v3_ci95_hi"])],
                        "V3 - raw-RF [95% CI]": [_ci(m, l, h) for m, l, h in zip(cu["delta_v3_minus_rawrf"], cu["delta_ci95_lo"], cu["delta_ci95_hi"])],
                        "Step change": [_f(v) for v in cu["step_change"]],
                        "Inflation vs leak-free": [_f(v) for v in cu["inflation_vs_clean"]],
                        "Inert": ["" if pd.isna(v) else ("yes" if str(v).lower() == "true" else "") for v in cu["inert"]]})
oa = pd.read_csv(os.path.join(_V14, "task2_one_at_a_time.csv"))
oat_fmt = pd.DataFrame({"State": oa["state"],
                        "V3 mean rho [95% CI]": [_ci(m, l, h) for m, l, h in zip(oa["v3_mean_rho"], oa["v3_ci95_lo"], oa["v3_ci95_hi"])],
                        "V3 - raw-RF [95% CI]": [_ci(m, l, h) for m, l, h in zip(oa["delta_v3_minus_rawrf"], oa["delta_ci95_lo"], oa["delta_ci95_hi"])],
                        "Standalone inflation": [_f(v) for v in oa["inflation_vs_clean"]]})
t4 = pd.read_csv(os.path.join(_V14, "task2_table4_inflation_column.csv"))
t4_fmt = pd.DataFrame({"Leak": [f"{_PLAIN[k]}" for k in t4["leak"]],
                       "Standalone inflation of mean rho": [_f(v) for v in t4["standalone_inflation"]], "Definition": t4["note"]})
sw = pd.read_csv(os.path.join(_V14, "exploratory_target_swap_shift.csv"))
swap_fmt = pd.DataFrame({"Comparison": sw["comparison"], "n": sw["n"],
                         "Mean shift [95% CI]": [_ci(m, l, h) for m, l, h in zip(sw["mean_shift"], sw["ci95_lo"], sw["ci95_hi"])],
                         "Positive/n": [f"{a}/{b}" for a, b in zip(sw["n_positive"], sw["n"])],
                         "P": [_f(v) for v in sw["p"]], "q (BH)": [_f(v) for v in sw["q"]]})
_out = {"task1_table2_formatted.csv": table2_fmt, "task1_absolute_rho_formatted.csv": abs_fmt,
        "task2_waterfall_cumulative_formatted.csv": cum_fmt, "task2_one_at_a_time_formatted.csv": oat_fmt,
        "task2_table4_inflation_column_formatted.csv": t4_fmt, "exploratory_target_swap_shift_formatted.csv": swap_fmt}
for _name, _df in _out.items():
    _df.to_csv(os.path.join(_V14, _name), index=False)
say(f"7. REPORT TABLES written to {_ND} decimals: " + ", ".join(_out))
for _name, _df in _out.items():
    print("\n" + _name); print(_df.to_string(index=False))

say("")
say("HAND-IN CHECK " + ("PASSED: every check above passed" if not fails else f"FAILED: {fails}"))
with open(os.path.join(_V14, "handin_check.txt"), "w") as f:
    f.write("\n".join(report) + "\n")